## Model Problem A: A Simple Statistical Inversion in 2 Dimensions

We consider the statistical inversion problem of estimating $(x,y) \in \mathbb{R}^2$ from data $z \in \mathbb{R}$ gathered according the measurement model:
$$
    z = f(x,y) + \eta \qquad \text{ where } \eta \sim N(0, \sigma^2) \text{ and } f: \mathbb{R}^2 \to \mathbb{R}
    \text{ is the forward function. }
    \tag{MM1}
$$
Leaving aside the measurement noise $\eta$ we would like to recover $(x,y) = f^{-1}(z)$.  This typically be impossible in priciple.  However, if we put a Gaussian prior $\mu_0 \sim N(0, C)$ on $(x,y)$ then the posterior 'bayesian solution' $(x,y)|z$ is the distribution:
$$
    \mu(dx, dy) \propto \exp\left( - \frac{1}{2 \sigma^2} ( f(x,y) - z)^2 
    - \frac{1}{2} <C^{-1}(x,y), (x,y)>\right) dx dy.
    \tag{S1}
$$

## Model Problem B: Matrix Coefficents from Solutions

We next consider inverse problem of estimating the coefficents of an $n \times n$ anti-semetric matrix $A$ from data $\theta \in \mathbb{R}^n$ gathered according the measurement model:
$$
    z = \mathcal{O}(\theta(A)) + \eta \qquad \text{ where } \eta \sim N(0, \sigma^2)
    \tag{MM2}
$$
Here $\theta := \theta(A)$ is the solution of
$$
    (A + \kappa I)\theta = g
$$
and where $\mathcal{O}(\theta)$ defined by $\mathcal{O}: \mathbb{R}^n \to \mathbb{R}^m$ is a partial observation of $\theta$. Typically
$$
    \mathcal{O}((\theta_1, \ldots, \theta_n)) = (\theta_1,\ldots, \theta_m)
$$
Here the Bayesian posterior will take the form
$$
    \mu(dA) \propto \exp\left( - \frac{1}{2 \sigma^2} 
    | \mathcal{O}( (A +\kappa I)^{-1} g) - z|^2 
    - \frac{1}{2} <C^{-1}A, A>\right) dA.
    \tag{S1}
$$
which is a target in $d = n(n-1)/2$


In [1]:
#Set-up: Packages to Import and Untility Functions

#Core Numerical Packages for Paral
import MCMC_Sampliers as MCMCsmp

#Numerical Elements
from numpy.linalg import norm
import numpy as np
from numpy import dot, array, transpose, diag
import random
import math

#Input Output utils
import os
import pandas as pd

#Stats elements
from scipy.stats import norm

#Plotting stuff
from matplotlib.patches import Ellipse

def writeCSV(filenm, sampArray, newFile = False):
    if newFile:
        pd.DataFrame(sampArray).to_csv(filenm, mode='w', index=False, header=False)
    else:
        pd.DataFrame(sampArray).to_csv(filenm, mode='a', index=False, header=False)

def readCSV(filenm):   
    df = pd.read_csv(filenm)
    MpCNsamp2 = df.to_numpy()
    return MpCNsamp2

#Fun Progress Bar
from tqdm.notebook import tqdm

#Parallel executation functions
from functools import partial
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp
import inspect
print("Current cpu count:" + str(mp.cpu_count()))

#Histogram and Other Graphics Utililities
import matplotlib.pyplot as plt
import matplotlib as mpl

def getComp(sampLst,indx):
    return [item[indx] for item in sampLst]

def makeHistGrid(R, dr, sampList, NumParams,saveLoc, C=None, beta=0.95, hidePlt = True):
    SampLstsGd = [getComp(sampList,j) for j in range(0,NumParams)]
    numbins= int(2*R/dr)
    x_bins = np.linspace(-R, R, numbins) 
    y_bins = np.linspace(-R, R, numbins)

    fig, axs = plt.subplots(NumParams, NumParams,figsize=(15,15))
    for i in range(0,NumParams):
        for j in range(0,NumParams):
            if i == j:
                axs[i,j].hist(SampLstsGd[i], density=True, bins=x_bins)
                # Add confidence interval ticks if covariance provided
                if C is not None:
                    # Quantile of standard normal
                    z = norm.ppf((1 + beta) / 2)
                    # Confidence interval endpoints
                    bound = z * np.sqrt(C[i, i])
                    # Get current y-axis limits for tick height
                    ylim = axs[i, j].get_ylim()
                    tick_height = 0.08 * (ylim[1] - ylim[0])
                    # Draw ticks at -bound and +bound
                    for x_pos in [-bound, bound]:
                        axs[i, j].axvline(x=x_pos, ymin=0, ymax=0.08, color='red', linewidth=2.5)   
                    # Add a small label
                    #axs[i, j].text(0, ylim[1] * 0.92, f'{100*beta:.0f}% CI', ha='center', va='top', fontsize=8, color='red')
            else:
                axs[i,j].hist2d(SampLstsGd[j],SampLstsGd[i],bins= [x_bins,y_bins])
                if C is not None:
                    # Extract 2x2 marginal covariance (note: hist2d has j on x-axis, i on y-axis)
                    Sigma = np.array([[C[j, j], C[j, i]],[C[i, j], C[i, i]]])
                    # Chi-squared quantile with 2 degrees of freedom
                    # For chi^2_2: quantile = -2 * ln(1 - beta)
                    chi2_val = -2 * np.log(1 - beta)
                    # Eigendecomposition of Sigma
                    eigenvalues, eigenvectors = np.linalg.eigh(Sigma)
    
                    # Sort by eigenvalue (largest first) for consistent orientation
                    order = eigenvalues.argsort()[::-1]
                    eigenvalues = eigenvalues[order]
                    eigenvectors = eigenvectors[:, order]
    
                    # Semi-axis lengths: sqrt(eigenvalue * chi2_quantile)
                    width = 2 * np.sqrt(eigenvalues[0] * chi2_val)
                    height = 2 * np.sqrt(eigenvalues[1] * chi2_val)
                    # Rotation angle (in degrees) from the first eigenvector
                    angle = np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0]))
    
                    # Create and add the ellipse patch
                    ellipse = Ellipse(xy=(0, 0), width=width, height=height, angle=angle,
                                      edgecolor='red', facecolor='none', linewidth=2, 
                                      linestyle='-', zorder=10)
                    axs[i,j].add_patch(ellipse)
        for ax in axs.flat:
            ax.label_outer()
    plt.savefig(saveLoc)
    if hidePlt:
        plt.close()


def plot_timeseries(samples, filename, burn_in=0):
    """
    Plot time series (trace plots) for each component of the MCMC chain
    and save to a file.

    Parameters
    ----------
    samples : np.ndarray
        Array of shape (L+1, dim) returned by MpCN. Each row is an iteration,
        each column is a component.
    filename : str
        Path (including extension, e.g. 'traceplots.png' or 'traceplots.pdf')
        where the figure will be saved.
    burn_in : int, optional
        Number of initial samples to discard before plotting.
    """
    # Ensure numpy array
    samples = np.asarray(samples)
    n_iter, dim = samples.shape

    # Indices of iterations after burn-in
    iters = np.arange(burn_in, n_iter)

    # Make one subplot per dimension
    fig, axes = plt.subplots(dim, 1, figsize=(8, 2.0 * dim), sharex=True)

    # If dim == 1, axes is not a list
    if dim == 1:
        axes = [axes]

    for d in range(dim):
        axes[d].plot(iters, samples[burn_in:, d])
        axes[d].set_ylabel(f"$q_{d+1}$")
        axes[d].grid(alpha=0.3)

    axes[-1].set_xlabel("Iteration")

    fig.suptitle("MpCN Trace Plots", y=0.99)
    fig.tight_layout()

    # Ensure directory exists (if any directory is specified)
    directory = os.path.dirname(filename)
    if directory:
        os.makedirs(directory, exist_ok=True)

    # Save and close
    fig.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close(fig)




def generate_colors(n, cmap_name="tab10"):
    """
    Return a list of `n` hex colors taken at equal intervals
    from a Matplotlib colormap (default: 'tab10').

    Parameters
    ----------
    n : int
        Number of colors you need.
    cmap_name : str, optional
        Any valid Matplotlib colormap name.  Sequential maps
        ('viridis', 'plasma', …) or qualitative maps ('tab10',
        'Set3', …) both work.

    Returns
    -------
    list[str]
        Hex strings like '#1f77b4' ready for plotting.
    """
    # Get a ListedColormap with exactly n entries
    cmap = plt.get_cmap(cmap_name, n)
    # Convert RGBA to hex for convenience
    return [mpl.colors.to_hex(cmap(i)) for i in range(cmap.N)]




Current cpu count:11


In [2]:
#Experiment 6

#Gaussian target (changing Covariance/mean on `low modes')

#Model dim
TarDim6 = 5

cov0 = 4
gam = 2
CovDiag_p = [cov0* (j**(-gam)) for j in list(range(1,TarDim6+1))]
#Q = MCMCsmp.rndm_orth_matrix(NumParmsEx4)
#CovEx4 = Q.T@ MCMCsmp.mkDiagCov(CovDiag)@ Q
CovPriorEx6 = MCMCsmp.mkDiagCov(CovDiag_p)
print(CovPriorEx6)

#Perturb dim
pertDim6 = 2
#CovDiag_post = [.1*CovDiag_p[0], 10*CovDiag_p[1]]
CovDiag_post = [CovDiag_p[1], CovDiag_p[0]] + CovDiag_p[pertDim6:]
CovPostEx6 = MCMCsmp.mkDiagCov(CovDiag_post)
print(CovPostEx6)

meanPostEx6 = np.zeros(pertDim6)


#Make Problem Information File
expId =  7
FileNmBase = "Data/Large_p_study/Experiment_6/" + "Ex_ID_"+ str(expId) + "/"
os.makedirs(FileNmBase, exist_ok=True)

probDataFile = FileNmBase + "Problem_Info.txt"
with open(probDataFile, 'w') as file:
    file.write("Target Type:  Model Problem C \n")
    file.write("Target Dimension: " + str(TarDim6) + "\n")
    file.write("Prior Covariance: " + str(CovPriorEx6) + "\n")
    file.write("Perturbation Dimension: " + str(pertDim6) + "\n")
    file.write("Posterior Mean: " + str(meanPostEx6) + "\n")
    file.write("Posterior Covariance: " + str(CovPostEx6)+ "\n")

PriorCovInvEx6 = MCMCsmp.mkDiagCov([CovDiag_p[0]**(-1),CovDiag_p[1]**(-1)])
PostCovInvEx6 = MCMCsmp.mkDiagCov([CovDiag_post[0]**(-1),CovDiag_post[1]**(-1)])

PotEx6Pass = partial(MCMCsmp.PotGaussPert, TarDim=TarDim6, PertDim=pertDim6, PriorCovInv=PriorCovInvEx6, PostMean=meanPostEx6, PostCovInv=PostCovInvEx6, mode = "soft")

#Testing Potential
#print(PostCovInvEx6 - PriorCovInvEx6)   
#priorSmpLst = np.random.multivariate_normal(np.zeros(TarDim6), CovPriorEx6, size=5)
#print([MCMCsmp.PotGaussPert(priorSmp,TarDim6,pertDim6, PriorCovInvEx6,meanPostEx6,PostCovInvEx6) for priorSmp in priorSmpLst])
#print([-1/2*x[0:pertDim6].T @(PostCovInvEx6- PriorCovInvEx6)@x[0:pertDim6]  for x in priorSmpLst])
#CovPriorEx6Ext = MCMCsmp.mkDiagCov(CovDiag_post + CovDiag_p[pertDim6:])
#print(CovPriorEx6)
#print(CovPriorEx6Ext)
#print([-1/2*x.T @(np.linalg.inv(CovPriorEx6Ext)- np.linalg.inv(CovPriorEx6))@x  for x in priorSmpLst])


[[4.         0.         0.         0.         0.        ]
 [0.         1.         0.         0.         0.        ]
 [0.         0.         0.44444444 0.         0.        ]
 [0.         0.         0.         0.25       0.        ]
 [0.         0.         0.         0.         0.16      ]]
[[1.         0.         0.         0.         0.        ]
 [0.         4.         0.         0.         0.        ]
 [0.         0.         0.44444444 0.         0.        ]
 [0.         0.         0.         0.25       0.        ]
 [0.         0.         0.         0.         0.16      ]]


In [3]:
rhotest =.5
ptest = 50
#lag = 200
invChainLn = 5000
numChains = 50

if __name__ == "__main__": 
    with ProcessPoolExecutor(max_workers=mp.cpu_count()) as pool:
        MCMCsampRun = []
        for run in range(0,numChains):
            q0z = np.random.multivariate_normal(np.zeros(TarDim6), CovPostEx6, size=1)

            CurfNinput = (q0z,TarDim6,CovPriorEx6,rhotest,PotEx6Pass,ptest,invChainLn)
            MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsmPCN,*CurfNinput))

        print("Total MCMC Runs: " + str(len(MCMCsampRun)))
        results = [None]*len(MCMCsampRun)
        # map Future -> index so we can keep ordered results
        where = {f:i for i, f in enumerate(MCMCsampRun)}

        for f in tqdm(as_completed(MCMCsampRun), total=len(MCMCsampRun), desc="Parallel MCMC Runs"):
            i = where[f]
            results[i] = f.result()


ESS_Data = []
for curRnInx in tqdm(range(0,numChains-1), desc= "Compiling Data"):
    ESS_Data.append(MCMCsampRun[curRnInx].result()[0][0])

# Make a histogram of the ESS_Data
plt.figure()
plt.hist(ESS_Data, bins='auto', edgecolor='black')
plt.xlabel('Value')
plt.ylabel('Count')
plt.title('Histogram of Computed ESS')
plt.grid(alpha=0.3)
plt.show()

# Compute the running average A = [y_1, ..., y_n]
A = []
running_sum = 0.0
for k, x in enumerate(ESS_Data, start=1):  # k goes 1..n
    running_sum += x
    y_k = running_sum / k          # y_k = (1/k) * sum_{j=1}^k x_j
    A.append(y_k)

# Plot y_k vs k
k_values = list(range(1, len(ESS_Data) + 1))

plt.figure()
plt.plot(k_values, A, marker='o')
plt.xlabel('k')
plt.ylabel('y_k (running average)')
plt.title('Running average vs index k')
plt.grid(True)
plt.show()

Total MCMC Runs: 50


Parallel MCMC Runs:   0%|          | 0/50 [00:00<?, ?it/s]

TypeError: mixMetricsmPCN() missing 5 required positional arguments: 'ESSruns', 'Nave', 'numRuns', 'numMom', and 'saveLn'

In [ ]:
# Experiment 6: rho v p

csvFileNm = FileNmBase + "Baseline_Samples.csv"

#Input to studies
#[p,NumRho,NumSamps,numChains]
#p: value to p to run study
#NumRho: 1/NumRho Specifies the step size for the study
#NumSamps: Length of the MCMC at each rho value

SampleLnSv = 2000

#ImpLst = [[10,10,5000,10]]
#ImpLst = [[10,500,500000,20],[100,100,200000,20],[1000,100,100000,10]]
ImpLst = [[10,100,100000,10],[100,100,50000,10]]

for Imp in ImpLst:
    pCur = Imp[0]

    NumRho = Imp[1]
    delRho = 1/NumRho
    rho = delRho
    NumSamps = Imp[2]
    NumChains = Imp[3]
    
    print("Currently running: p=" + str(pCur))
    print("Delta rho: " + str(delRho))
    print("Number of Samples: " + str(NumSamps))

    rhoLst = []
    ESSLstOG = []
    ESSLstLoc = []
    ESSLstGlob = []
    MSJDLstOG = []
    MSJDLstLoc = []
    MSJDLstGlob = []

    for parmIndx in range(0,TarDim6):
        ESSLstOG.append([])
        ESSLstLoc.append([])
        ESSLstGlob.append([])

    rhoCur= rho
    for curRnInx in range(0,NumRho-1):
        rhoLst.append(rhoCur)
        rhoCur += delRho

    if __name__ == "__main__": 
        MCMCsampRun = []
        with ProcessPoolExecutor(max_workers=mp.cpu_count()) as pool:
            for rhoCur in rhoLst:
                q0zSt = np.random.multivariate_normal(np.zeros(TarDim6), CovPostEx6, size=NumChains)

                CurfNinput = (q0zSt,TarDim6,CovPriorEx6,rhoCur,PotEx6Pass,pCur,NumSamps,NumChains,SampleLnSv)
                MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsmPCNMulti,*CurfNinput))
                MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsmlocPCNMTMMulti,*CurfNinput))
                MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsmPCNMTMMulti,*CurfNinput))

            print("Total MCMC Runs: " + str(len(MCMCsampRun)))
            results = [None]*len(MCMCsampRun)
            # map Future -> index so we can keep ordered results
            where = {f:i for i, f in enumerate(MCMCsampRun)}

            for f in tqdm(as_completed(MCMCsampRun), total=len(MCMCsampRun), desc="Parallel MCMC Runs"):
                i = where[f]
                results[i] = f.result()
    
    
    for curRnInx in tqdm(range(0,NumRho-1), desc= "Compiling Data"):
        curSampOGData = MCMCsampRun[3*curRnInx].result()
        curSampLocData = MCMCsampRun[3*curRnInx +1].result()
        curSampGlobData = MCMCsampRun[3*curRnInx +2].result()

        for parmIndx in range(0,TarDim6):    
            ESSLstOG[parmIndx].append(curSampOGData[0][parmIndx])
            ESSLstLoc[parmIndx].append(curSampLocData[0][parmIndx])
            ESSLstGlob[parmIndx].append(curSampGlobData[0][parmIndx])
        
        MSJDLstOG.append(curSampOGData[1])
        MSJDLstLoc.append(curSampLocData[1])
        MSJDLstGlob.append(curSampGlobData[1])

        curRunDataTS= "time_series_p_" + str(pCur) + "/rho_" + str(curRnInx*(delRho +1))
        plot_timeseries(curSampOGData[2], FileNmBase + curRunDataTS+ "_mpCN.png", burn_in=1000)
        plot_timeseries(curSampLocData[2], FileNmBase + curRunDataTS+ "_mpCN_Loc.png", burn_in=1000)
        plot_timeseries(curSampLocData[2], FileNmBase + curRunDataTS+ "_mpCN_Glob.png", burn_in=1000)

    curRunData ="p_" + str(pCur) + "_drho_" + str(delRho) + "_NSamps_" + str(NumSamps) + "_NChains_" + str(NumChains)
    csvFileNm = FileNmBase + curRunData +  "_DATA.csv"

    writeCSV(csvFileNm,[rhoLst,ESSLstOG,ESSLstLoc,ESSLstGlob,MSJDLstOG,MSJDLstLoc,MSJDLstGlob])
    

    
    #Generate figures for ESS/N vs rho for each component of the posterior
    for parmIndx in range(0,TarDim6):
        fig, ax = plt.subplots(figsize=(6, 4))

        ax.plot(rhoLst, ESSLstOG[parmIndx], linestyle="-", label=r"$mpCN$", color="tab:orange")
        ax.plot(rhoLst, ESSLstLoc[parmIndx], linestyle="-", label=r"$mpCNlocMTM$", color="tab:green")
        ax.plot(rhoLst, ESSLstGlob[parmIndx], linestyle="-",label=r"$mpCNMTM$", color="tab:blue")

        ax.set_xlabel(r"$\rho$")
        ax.set_ylabel(r"$ESS/N$")
        ax.set_title(r"Mixing as measured by ESS/N for p ="+str(pCur)+" for Parameter Index " + str(parmIndx))
        ax.grid(alpha=0.3)
        ax.legend()

        plt.tight_layout()
        plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_ParaIndx_" + str(parmIndx)+ ".png")
        plt.close(fig)
    
    
    fig_colors = generate_colors(TarDim6)
    
    #Generate figures for ESS/N comparing different components for mpCN
    fig, ax = plt.subplots(figsize=(6, 4))
    for parmIndx in range(0,TarDim6):
        ax.plot(rhoLst, ESSLstOG[parmIndx], linestyle="-", label="cmp_"+str(parmIndx), color=fig_colors[parmIndx])
    
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(r"$ESS/N$")
    ax.set_title(r"Mixing for mpCN as measured by ESS/N for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_mpCN.png")
    plt.close(fig)
    
    #Generate figures for ESS/N comparing different components for mpCNMTM
    fig, ax = plt.subplots(figsize=(6, 4))
    for parmIndx in range(0,TarDim6):
        ax.plot(rhoLst, ESSLstGlob[parmIndx], linestyle="-", label="cmp_"+str(parmIndx), color=fig_colors[parmIndx])
    
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(r"$ESS/N$")
    ax.set_title(r"Mixing for mpCN_MTM as measured by ESS/N for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_mpCN_MTM.png")
    plt.close(fig)
    
    #Generate figures for ESS/N comparing different components for mpCNMTMloc
    fig, ax = plt.subplots(figsize=(6, 4))
    for parmIndx in range(0,TarDim6):
        ax.plot(rhoLst, ESSLstLoc[parmIndx], linestyle="-", label="cmp_"+str(parmIndx), color=fig_colors[parmIndx])
    
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(r"$ESS/N$")
    ax.set_title(r"Mixing for mpCN_loc_MTM as measured by ESS/N for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_mpCN_MTM_loc.png")
    plt.close(fig)
    
    fig, ax = plt.subplots(figsize=(6, 4))

    ax.plot(rhoLst, MSJDLstOG, linestyle="-",label=r"$mpCN$", color="tab:orange")
    ax.plot(rhoLst, MSJDLstLoc, linestyle="-",label=r"$mpCNlocMTM$", color="tab:green")
    ax.plot(rhoLst, MSJDLstGlob, linestyle="-",label=r"$mpCNMTM$", color="tab:blue")

    ax.set_xlabel(r"rho")
    ax.set_ylabel(r"MSJD")
    ax.set_title(r"Mixing as measured by MSJD for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_MSJD_v_rho.png")
    plt.close(fig)

In [ ]:
#Experiment 1: Set-up

#We consider a target of the type Model Problem A, [MM1] where
#    f(x,y) = y(x-a)^r 

#The problem parameters are:
expId = 1 #Experiment ID

NumParmsEx1 = 2
zEx1 = 6
aEx1 = .3
rEx1 = 1
sigEx1 = 1
CovEx1 = np.diag([3,2])

#Make Loglikehood function with lambda workaround
#using PotEx1(X,sig, a,r,z)
#print(inspect.signature(MCMCsmp.PotEx1))

PotEx1Pass = partial(MCMCsmp.PotEx1, sig=sigEx1, a= aEx1, r=rEx1, z=zEx1,  mode="soft")


#Make Problem Information File
FileNmBase = "Data/Large_p_study/Experiment_1/" + "Ex_ID_"+ str(expId) + "/"
os.makedirs(FileNmBase, exist_ok=True)

probDataFile = FileNmBase + "Problem_Info.txt"
fFnStr = "y (x - " + str(.3) + ")^" + str(rEx1) 

with open(probDataFile, 'w') as file:
    file.write("Target Type:  Model Problem A \n")
    file.write("Model Dim: " + str(NumParmsEx1) + "\n")
    file.write("Forward Map:  f(x,y) = " + fFnStr + "\n")
    file.write("z: " + str(zEx1) + "\n")
    file.write("sig: " + str(sigEx1)+ "\n")
    file.write("Prior Cov: " + str(CovEx1))

L = 1000000
stffMEx1, stffVEx1 = MCMCsmp.DetStiffness(NumParmsEx1,PotEx1Pass,CovEx1,L)
#for k in range(0,5):
#    stffMEx1, stffVEx1 = MCMCsmp.DetStiffness(NumParmsEx1,PotEx1Pass,CovEx1,L)
#    print("Mean: " + str(stffMEx1))
#    print("Var: " + str(stffVEx1) + "\n")


with open(probDataFile, 'a') as file:
    file.write("\n")
    file.write("Problem Stiffness: \n")
    file.write("M := int Pot(x) mu0(dx): " + str(stffMEx1) + "\n")
    file.write("V := int (Pot(x) -M)^2 mu0(dx): " + str(stffVEx1))



In [ ]:
#Experiment 1:
#Perform a baseline/warm-up run


q0 = np.zeros(NumParmsEx1)
rhoWarm = .01
pWarm = 100

numSmpWarm = 5000
burninWarm = 4000

WarmSamps = MCMCsmp.MpCN(q0,NumParmsEx1,CovEx1,rhoWarm,PotEx1Pass,pWarm,numSmpWarm)

NumRuns =  100 #of total runs
NumSamps = 10000 #samples per run

if __name__ == "__main__": 
    
    with ProcessPoolExecutor(max_workers=mp.cpu_count()) as pool:
        MCMCsampRun = []
        for run in range(0,NumRuns):
            strIndx = random.randint(burninWarm, numSmpWarm)
            q0zCur = WarmSamps[strIndx]
            CurfNinput = (q0zCur,NumParmsEx1,CovEx1,rhoWarm,PotEx1Pass,pWarm,NumSamps)
            MCMCsampRun.append(pool.submit(MCMCsmp.MpCN,*CurfNinput))


        print("Total MCMC Runs: " + str(len(MCMCsampRun)))
        results = [None]*len(MCMCsampRun)
        # map Future -> index so we can keep ordered results
        where = {f:i for i, f in enumerate(MCMCsampRun)}

        for f in tqdm(as_completed(MCMCsampRun), total=len(MCMCsampRun), desc="Parallel MCMC Runs"):
            i = where[f]
            results[i] = f.result()
    csvFileNm = FileNmBase + "Baseline_Samples.csv"
    for curRnInx in tqdm(range(0,len(MCMCsampRun)), desc= "Compiling Data"):
         writeCSV(csvFileNm,MCMCsampRun[curRnInx].result())
               

In [ ]:
#Generate Histogram
#Dimensions For Histogram Plot

#The problem parameters are:
expId = 1 #Experiment ID

#Make Problem Information File
FileNmBase = "Data/Large_p_study/Experiment_1/" + "Ex_ID_"+ str(expId) + "/"
os.makedirs(FileNmBase, exist_ok=True)

CovEx1 = np.diag([3,2])
NumParmsEx1 = 2

csvFileNm = FileNmBase + "Baseline_Samples.csv"

ex1MCMCSmpBase = readCSV(csvFileNm)
MCMCpoolLen = len(ex1MCMCSmpBase)
print("Number of MpCN Samples Now Available: " + str(MCMCpoolLen))

R = 5
dr = .1

histFileNm = FileNmBase + "Baseline_Histogram.png"
makeHistGrid(R, dr, ex1MCMCSmpBase, NumParmsEx1,histFileNm, CovEx1, .95, True)

In [ ]:
#Experiment 1: rho v p

#Input to studies
#[p,NumRho,NumSamples]

#ImpLst = [[10,50,500000],[100,50,500000],[1000,20,500000],[10000,20,100000]]
ImpLst = [[10,20,10000],[100,20,10000],[1000,20,5000],[10000,10,1000]]
#ImpLst = [[10,100,200]]
#ImpLst = [[10,10,1000],[100,10,1000]]


for Imp in ImpLst:
    pCur = Imp[0]

    NumRho = Imp[1]
    delRho = 1/NumRho
    rho = delRho
    NumSamps = Imp[2]
    
    print("Currently running: p=" + str(pCur))
    print("Delta rho: " + str(delRho))
    print("Number of Samples: " + str(NumSamps))

    rhoLst = []
    ESSLstOG = []
    ESSLstLoc = []
    ESSLstGlob = []
    MSJDLstOG = []
    MSJDLstLoc = []
    MSJDLstGlob = []

    for parmIndx in range(0,NumParmsEx1):
        ESSLstOG.append([])
        ESSLstLoc.append([])
        ESSLstGlob.append([])

    rhoCur= rho
    for curRnInx in range(0,NumRho-1):
        rhoLst.append(rhoCur)
        rhoCur += delRho

    if __name__ == "__main__": 
        MCMCsampRun = []
        with ProcessPoolExecutor(max_workers=mp.cpu_count()) as pool:
            for rhoCur in rhoLst:
                strIndx = random.randint(math.floor(.95* MCMCpoolLen), MCMCpoolLen)
                q0z = ex1MCMCSmpBase[strIndx]
                
                CurfNinput = (q0z,NumParmsEx1,CovEx1,rhoCur,PotEx1Pass,pCur,NumSamps)
                MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsmPCN,*CurfNinput))
                MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsmlocPCNMTM,*CurfNinput))
                MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsBBMTM,*CurfNinput))

            print("Total MCMC Runs: " + str(len(MCMCsampRun)))
            results = [None]*len(MCMCsampRun)
            # map Future -> index so we can keep ordered results
            where = {f:i for i, f in enumerate(MCMCsampRun)}

            for f in tqdm(as_completed(MCMCsampRun), total=len(MCMCsampRun), desc="Parallel MCMC Runs"):
                i = where[f]
                results[i] = f.result()
    
    
    for curRnInx in tqdm(range(0,NumRho-1), desc= "Compiling Data"):
        curSampOGData = MCMCsampRun[3*curRnInx].result()
        curSampLocData = MCMCsampRun[3*curRnInx +1].result()
        curSampGlobData = MCMCsampRun[3*curRnInx +2].result()

        for parmIndx in range(0,NumParmsEx1):    
            ESSLstOG[parmIndx].append(curSampOGData[0][parmIndx])
            ESSLstLoc[parmIndx].append(curSampLocData[0][parmIndx])
            ESSLstGlob[parmIndx].append(curSampGlobData[0][parmIndx])
        
        MSJDLstOG.append(curSampOGData[1])
        MSJDLstLoc.append(curSampLocData[1])
        MSJDLstGlob.append(curSampGlobData[1])

    curRunData ="p_" + str(pCur) + "_drho_" + str(delRho) + "_NSamps_" + str(NumSamps)
    csvFileNm = FileNmBase + curRunData +  "_DATA.csv"

    writeCSV(csvFileNm,[rhoLst,ESSLstOG,ESSLstLoc,ESSLstGlob,MSJDLstOG,MSJDLstLoc,MSJDLstGlob])
    

    
    #Generate figures for ESS/N vs rho for each component of the posterior
    for parmIndx in range(0,NumParmsEx1):
        fig, ax = plt.subplots(figsize=(6, 4))

        ax.plot(rhoLst, ESSLstOG[parmIndx], linestyle="-", label=r"$mpCN$", color="tab:orange")
        ax.plot(rhoLst, ESSLstLoc[parmIndx], linestyle="-", label=r"$mpCNlocMTM$", color="tab:green")
        ax.plot(rhoLst, ESSLstGlob[parmIndx], linestyle="-",label=r"$mpCNMTM$", color="tab:blue")

        ax.set_xlabel(r"$\rho$")
        ax.set_ylabel(r"$ESS/N$")
        ax.set_title(r"Mixing as measured by ESS/N for p ="+str(pCur)+" for Parameter Index " + str(parmIndx))
        ax.grid(alpha=0.3)
        ax.legend()

        plt.tight_layout()
        plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_ParaIndx_" + str(parmIndx)+ ".png")
        plt.close(fig)
    
    
    fig_colors = generate_colors(NumParmsEx1)
    
    #Generate figures for ESS/N comparing different components for mpCN
    fig, ax = plt.subplots(figsize=(6, 4))
    for parmIndx in range(0,NumParmsEx1):
        ax.plot(rhoLst, ESSLstOG[parmIndx], linestyle="-", label="cmp_"+str(parmIndx), color=fig_colors[parmIndx])
    
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(r"$ESS/N$")
    ax.set_title(r"Mixing for mpCN as measured by ESS/N for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_mpCN.png")
    plt.close(fig)
    
    #Generate figures for ESS/N comparing different components for mpCNMTM
    fig, ax = plt.subplots(figsize=(6, 4))
    for parmIndx in range(0,NumParmsEx1):
        ax.plot(rhoLst, ESSLstGlob[parmIndx], linestyle="-", label="cmp_"+str(parmIndx), color=fig_colors[parmIndx])
    
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(r"$ESS/N$")
    ax.set_title(r"Mixing for mpCN_MTM as measured by ESS/N for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_mpCN_MTM.png")
    plt.close(fig)
    
    #Generate figures for ESS/N comparing different components for mpCNMTMloc
    fig, ax = plt.subplots(figsize=(6, 4))
    for parmIndx in range(0,NumParmsEx1):
        ax.plot(rhoLst, ESSLstLoc[parmIndx], linestyle="-", label="cmp_"+str(parmIndx), color=fig_colors[parmIndx])
    
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(r"$ESS/N$")
    ax.set_title(r"Mixing for mpCN_loc_MTM as measured by ESS/N for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_mpCN_MTM_loc.png")
    plt.close(fig)
    
    fig, ax = plt.subplots(figsize=(6, 4))

    ax.plot(rhoLst, MSJDLstOG, linestyle="-",label=r"$mpCN$", color="tab:orange")
    ax.plot(rhoLst, MSJDLstLoc, linestyle="-",label=r"$mpCNlocMTM$", color="tab:green")
    ax.plot(rhoLst, MSJDLstGlob, linestyle="-",label=r"$mpCNMTM$", color="tab:blue")

    ax.set_xlabel(r"rho")
    ax.set_ylabel(r"MSJD")
    ax.set_title(r"Mixing as measured by MSJD for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_MSJD_v_rho.png")
    plt.close(fig)


In [ ]:
#Experiment 2: Set-up

#We consider the inverse problem 
#f(x,y) = (x - x_0)^2/a^2 +  (y - y_0)^2/b^2

#The problem parameters are:
expId = 1 #Experiment ID

zdataEx2 = 2
sigEx2 = .05
a0Ex2 = 1
b0Ex2 = 3
asqIEx2 = a0Ex2**(-2) 
bsqIEx2 = b0Ex2**(-2)
x0Ex2 = 2
y0Ex2 = 2

fFnStrEx2 = "(x - " + str(x0Ex2) + ")^2*" + str(a0Ex2) + "^(-2) + (y - " + str(x0Ex2) + ")^2*" + str(b0Ex2) + "^(-2)"

NumParmsEx2 = 2

PriorCovEVs = [.5,2]
Tth = 2*np.pi/3

Rot = np.array([
        [np.cos(Tth), -np.sin(Tth)],
        [np.sin(Tth),  np.cos(Tth)]
    ])
CovEx2 = Rot.T @ np.diag(PriorCovEVs) @ Rot

#Make Problem Information File
expId = 1
FileNmBase = "Data/Large_p_study/Experiment_2/" + "Ex_ID_"+ str(expId) + "/"
os.makedirs(FileNmBase, exist_ok=True)

probDataFile = FileNmBase + "Problem_Info.txt"
with open(probDataFile, 'w') as file:
    file.write("Target Type:  Model Problem A \n")
    file.write("Model Dim: " + str(NumParmsEx2) + "\n")
    file.write("Forward Map:  f(x,y) = " + fFnStrEx2 + "\n")
    file.write("z: " + str(zdataEx2) + "\n")
    file.write("sig: " + str(sigEx2)+ "\n")
    file.write("Prior Cov: " + str(CovEx2))

#Make Loglikehood function with lambda workaround
#using PotEx2(X, x0, y0, asqI, bsqI, sig, zdata, mode = None): 
#print(inspect.signature(MCMCsmp.PotEx2))

PotEx2Pass = partial(MCMCsmp.PotEx2, x0 = x0Ex2, y0 = y0Ex2,asqI= asqIEx2, bsqI=bsqIEx2, sig=sigEx2, zdata= zdataEx2, mode="soft")

L = 100000
stffMEx2, stffVEx2 = MCMCsmp.DetStiffness(NumParmsEx2,PotEx2Pass,CovEx2,L)

with open(probDataFile, 'a') as file:
    file.write("\n")
    file.write("Problem Stiffness: \n")
    file.write("M := int Pot(x) mu0(dx): " + str(stffMEx2) + "\n")
    file.write("V := int (Pot(x) -M)^2 mu0(dx): " + str(stffVEx2))


          

In [ ]:
#Perform a baseline/warm-up run
q0 = np.zeros(NumParmsEx2)
rhoWarm = .1
pWarm = 100

numSmpWarm = 5000
burninWarm = 4000

WarmSamps = MCMCsmp.MpCN(q0,NumParmsEx2,CovEx2,rhoWarm,PotEx2Pass,pWarm,numSmpWarm)

NumRuns =  100 #of total runs
NumSamps = 10000 #samples per run

if __name__ == "__main__": 
    
    with ProcessPoolExecutor(max_workers=mp.cpu_count()) as pool:
        MCMCsampRun = []
        for run in range(0,NumRuns):
            strIndx = random.randint(burninWarm, numSmpWarm)
            q0zCur = WarmSamps[strIndx]
            CurfNinput = (q0zCur,NumParmsEx2,CovEx2,rhoWarm,PotEx2Pass,pWarm,NumSamps)
            MCMCsampRun.append(pool.submit(MCMCsmp.MpCN,*CurfNinput))


        print("Total MCMC Runs: " + str(len(MCMCsampRun)))
        results = [None]*len(MCMCsampRun)
        # map Future -> index so we can keep ordered results
        where = {f:i for i, f in enumerate(MCMCsampRun)}

        for f in tqdm(as_completed(MCMCsampRun), total=len(MCMCsampRun), desc="Parallel MCMC Runs"):
            i = where[f]
            results[i] = f.result()
    csvFileNm = FileNmBase + "Baseline_Samples.csv"
    for curRnInx in tqdm(range(0,len(MCMCsampRun)), desc= "Compiling Data"):
         writeCSV(csvFileNm,MCMCsampRun[curRnInx].result())





In [ ]:
#Generate Histogram
#Dimensions For Histogram Plot



expId = 1
FileNmBase = "Data/Large_p_study/Experiment_2/" + "Ex_ID_"+ str(expId) + "/"
os.makedirs(FileNmBase, exist_ok=True)
csvFileNm = FileNmBase + "Baseline_Samples.csv"

ex2MCMCSmpBase = readCSV(csvFileNm)
MCMCpoolLen = len(ex2MCMCSmpBase)
print("Number of MpCN Samples Now Available: " + str(MCMCpoolLen))

NumParmsEx2 = 2

PriorCovEVs = [.5,2]
Tth = 2*np.pi/3

Rot = np.array([
        [np.cos(Tth), -np.sin(Tth)],
        [np.sin(Tth),  np.cos(Tth)]
    ])
CovEx2 = Rot.T @ np.diag(PriorCovEVs) @ Rot


R = 5
dr = .01

histFileNm = FileNmBase + "Baseline_Histogram.png"
makeHistGrid(R, dr, ex2MCMCSmpBase, NumParmsEx2,histFileNm, CovEx2, .95, True)

In [ ]:
#Experiment 2: rho v p

#Input to studies
#[p,NumRho,NumSamples]

#ImpLst = [[10,20,10000],[100,20,10000],[1000,20,5000],[10000,10,1000]]
#ImpLst = [[10,20,50000],[100,20,20000],[1000,20,10000],[10000,20,1000]]
ImpLst = [[1000,100,10000]]
#ImpLst = [[10,100,200]]
#ImpLst = [[10,10,1000],[100,10,1000]]


for Imp in ImpLst:
    pCur = Imp[0]

    NumRho = Imp[1]
    delRho = 1/NumRho
    rho = delRho
    NumSamps = Imp[2]
    
    print("Currently running: p=" + str(pCur))
    print("Delta rho: " + str(delRho))
    print("Number of Samples: " + str(NumSamps))

    rhoLst = []
    ESSLstOG = []
    ESSLstLoc = []
    ESSLstGlob = []
    MSJDLstOG = []
    MSJDLstLoc = []
    MSJDLstGlob = []

    for parmIndx in range(0,NumParmsEx2):
        ESSLstOG.append([])
        ESSLstLoc.append([])
        ESSLstGlob.append([])

    rhoCur= rho
    for curRnInx in range(0,NumRho-1):
        rhoLst.append(rhoCur)
        rhoCur += delRho

    if __name__ == "__main__": 
        MCMCsampRun = []
        with ProcessPoolExecutor(max_workers=mp.cpu_count()) as pool:
            for rhoCur in rhoLst:
                strIndx = random.randint(math.floor(.95* MCMCpoolLen), MCMCpoolLen)
                q0z = ex2MCMCSmpBase[strIndx]
                
                CurfNinput = (q0z,NumParmsEx2,CovEx2,rhoCur,PotEx2Pass,pCur,NumSamps)
                MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsmPCN,*CurfNinput))
                MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsmlocPCNMTM,*CurfNinput))
                MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsBBMTM,*CurfNinput))

            print("Total MCMC Runs: " + str(len(MCMCsampRun)))
            results = [None]*len(MCMCsampRun)
            # map Future -> index so we can keep ordered results
            where = {f:i for i, f in enumerate(MCMCsampRun)}

            for f in tqdm(as_completed(MCMCsampRun), total=len(MCMCsampRun), desc="Parallel MCMC Runs"):
                i = where[f]
                results[i] = f.result()
    
    
    for curRnInx in tqdm(range(0,NumRho-1), desc= "Compiling Data"):
        curSampOGData = MCMCsampRun[3*curRnInx].result()
        curSampLocData = MCMCsampRun[3*curRnInx +1].result()
        curSampGlobData = MCMCsampRun[3*curRnInx +2].result()

        for parmIndx in range(0,NumParmsEx2):    
            ESSLstOG[parmIndx].append(curSampOGData[0][parmIndx])
            ESSLstLoc[parmIndx].append(curSampLocData[0][parmIndx])
            ESSLstGlob[parmIndx].append(curSampGlobData[0][parmIndx])
        
        MSJDLstOG.append(curSampOGData[1])
        MSJDLstLoc.append(curSampLocData[1])
        MSJDLstGlob.append(curSampGlobData[1])

    curRunData ="p_" + str(pCur) + "_drho_" + str(delRho) + "_NSamps_" + str(NumSamps)
    csvFileNm = FileNmBase + curRunData +  "_DATA.csv"

    writeCSV(csvFileNm,[rhoLst,ESSLstOG,ESSLstLoc,ESSLstGlob,MSJDLstOG,MSJDLstLoc,MSJDLstGlob])
    

    
    #Generate figures for ESS/N vs rho for each component of the posterior
    for parmIndx in range(0,NumParmsEx2):
        fig, ax = plt.subplots(figsize=(6, 4))

        ax.plot(rhoLst, ESSLstOG[parmIndx], linestyle="-", label=r"$mpCN$", color="tab:orange")
        ax.plot(rhoLst, ESSLstLoc[parmIndx], linestyle="-", label=r"$mpCNlocMTM$", color="tab:green")
        ax.plot(rhoLst, ESSLstGlob[parmIndx], linestyle="-",label=r"$mpCNMTM$", color="tab:blue")

        ax.set_xlabel(r"$\rho$")
        ax.set_ylabel(r"$ESS/N$")
        ax.set_title(r"Mixing as measured by ESS/N for p ="+str(pCur)+" for Parameter Index " + str(parmIndx))
        ax.grid(alpha=0.3)
        ax.legend()

        plt.tight_layout()
        plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_ParaIndx_" + str(parmIndx)+ ".png")
        plt.close(fig)
    
    
    fig_colors = generate_colors(NumParmsEx2)
    
    #Generate figures for ESS/N comparing different components for mpCN
    fig, ax = plt.subplots(figsize=(6, 4))
    for parmIndx in range(0,NumParmsEx2):
        ax.plot(rhoLst, ESSLstOG[parmIndx], linestyle="-", label="cmp_"+str(parmIndx), color=fig_colors[parmIndx])
    
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(r"$ESS/N$")
    ax.set_title(r"Mixing for mpCN as measured by ESS/N for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_mpCN.png")
    plt.close(fig)
    
    #Generate figures for ESS/N comparing different components for mpCNMTM
    fig, ax = plt.subplots(figsize=(6, 4))
    for parmIndx in range(0,NumParmsEx2):
        ax.plot(rhoLst, ESSLstGlob[parmIndx], linestyle="-", label="cmp_"+str(parmIndx), color=fig_colors[parmIndx])
    
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(r"$ESS/N$")
    ax.set_title(r"Mixing for mpCN_MTM as measured by ESS/N for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_mpCN_MTM.png")
    plt.close(fig)
    
    #Generate figures for ESS/N comparing different components for mpCNMTMloc
    fig, ax = plt.subplots(figsize=(6, 4))
    for parmIndx in range(0,NumParmsEx2):
        ax.plot(rhoLst, ESSLstLoc[parmIndx], linestyle="-", label="cmp_"+str(parmIndx), color=fig_colors[parmIndx])
    
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(r"$ESS/N$")
    ax.set_title(r"Mixing for mpCN_loc_MTM as measured by ESS/N for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_mpCN_MTM_loc.png")
    plt.close(fig)
    
    fig, ax = plt.subplots(figsize=(6, 4))

    ax.plot(rhoLst, MSJDLstOG, linestyle="-",label=r"$mpCN$", color="tab:orange")
    ax.plot(rhoLst, MSJDLstLoc, linestyle="-",label=r"$mpCNlocMTM$", color="tab:green")
    ax.plot(rhoLst, MSJDLstGlob, linestyle="-",label=r"$mpCNMTM$", color="tab:blue")

    ax.set_xlabel(r"rho")
    ax.set_ylabel(r"MSJD")
    ax.set_title(r"Mixing as measured by MSJD for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_MSJD_v_rho.png")
    plt.close(fig)


In [ ]:
#Experiment 3: Set-up

#This experiment is based on Model Problem B

#We Specifying Problem Parameters as follows
#Model dimension and parameter space size are
ModDmEx3= 4
NumParmsEx3 = int(ModDmEx3*(ModDmEx3 -1)/2)

#We use precisely the example appearing in refer [1]
expId = 1 #Experiment ID
 
gvecEx3 = np.array([.1,0,5,2])
sigEx3 = 2
zEx3 = np.array([4.601,18.021,0,0])
dataDimEx3 = 2
kapEx3= .05

# Covariance of the 'prior' C = cov0[1^{-gam}, 2^[-gam],..., N^{-gam}]
# where N is the number of parameters in the model NumParms = 6 
#for number of enetries that need to be specified in A
cov0 = 5
gam = 1.5
CovEx3 = np.diag([cov0* (j**(-gam)) for j in list(range(1,NumParmsEx3+1))])

#Make Problem Information File
FileNmBase = "Data/Large_p_study/Experiment_3/" + "Ex_ID_"+ str(expId) + "/"
os.makedirs(FileNmBase, exist_ok=True)

probDataFile = FileNmBase + "Problem_Info.txt"
with open(probDataFile, 'w') as file:
    file.write("Target Type:  Model Problem B \n")
    file.write("Model Dim: " + str(NumParmsEx3) + "\n")
    file.write("g: " + str(gvecEx3) + "\n")
    file.write("z: " + str(zEx3[0:2]) + "\n")
    file.write("sig: " + str(sigEx3)+ "\n")
    file.write("kappa: " + str(kapEx3)+ "\n")
    file.write("Prior Cov: " + str(CovEx3))

#Make Loglikehood function with lambda workaround
#using PotExAD(a, gvec, sig, ModDm, z, kap, dataDim, mode = None):
#print(inspect.signature(MCMCsmp.PotExAD))


PotEx3Pass = partial(MCMCsmp.PotExAD, gvec=gvecEx3, sig=sigEx3, ModDm=ModDmEx3, z=zEx3, kap=kapEx3, dataDim=dataDimEx3, mode = "soft")

L = 100000
stffMEx3, stffVEx3 = MCMCsmp.DetStiffness(NumParmsEx3,PotEx3Pass,CovEx3,L)
#for k in range(0,10):
#    stffMEx3, stffVEx3 = MCMCsmp.DetStiffness(NumParmsEx3,PotEx3Pass,CovEx3,L)
#    print("Mean: " + str(stffMEx3))
#    print("Var: " + str(stffVEx3) + "\n")


with open(probDataFile, 'a') as file:
    file.write("\n")
    file.write("Problem Stiffness: \n")
    file.write("M := int Pot(x) mu0(dx): " + str(stffMEx3) + "\n")
    file.write("V := int (Pot(x) -M)^2 mu0(dx): " + str(stffVEx3))




          

In [ ]:
#Experiment 3:
#Perform a baseline/warm-up run
q0 = np.zeros(NumParmsEx3)
rhoWarm = .1
pWarm = 100

numSmpWarm = 5000
burninWarm = 4000

WarmSamps = MCMCsmp.MpCN(q0,NumParmsEx3,CovEx3,rhoWarm,PotEx3Pass,pWarm,numSmpWarm)

NumRuns =  10 #of total runs
NumSamps = 1000 #samples per run

if __name__ == "__main__": 
    
    with ProcessPoolExecutor(max_workers=mp.cpu_count()) as pool:
        MCMCsampRun = []
        for run in range(0,NumRuns):
            strIndx = random.randint(burninWarm, numSmpWarm)
            q0zCur = WarmSamps[strIndx]
            CurfNinput = (q0zCur,NumParmsEx3,CovEx3,rhoWarm,PotEx3Pass,pWarm,numSmpWarm)
            MCMCsampRun.append(pool.submit(MCMCsmp.MpCN,*CurfNinput))


        print("Total MCMC Runs: " + str(len(MCMCsampRun)))
        results = [None]*len(MCMCsampRun)
        # map Future -> index so we can keep ordered results
        where = {f:i for i, f in enumerate(MCMCsampRun)}

        for f in tqdm(as_completed(MCMCsampRun), total=len(MCMCsampRun), desc="Parallel MCMC Runs"):
            i = where[f]
            results[i] = f.result()
    csvFileNm = FileNmBase + "Baseline_Samples.csv"
    for curRnInx in tqdm(range(0,len(MCMCsampRun)), desc= "Compiling Data"):
         writeCSV(csvFileNm,MCMCsampRun[curRnInx].result())

#Generate Histogram
#Dimensions For Histogram Plot

ex3MCMCSmpBase = readCSV(csvFileNm)
MCMCpoolLen = len(ex3MCMCSmpBase)
print("Number of MpCN Samples Now Available: " + str(MCMCpoolLen))

R = 5
dr = .1

histFileNm = FileNmBase + "Baseline_Histogram.png"
makeHistGrid(R, dr, ex3MCMCSmpBase, NumParmsEx3,histFileNm, CovEx3, 0.95, True)

In [ ]:
##Generate Histograms of different rotations of the posterior

os.makedirs(FileNmBase + "/Rotations", exist_ok=True)

numRots = 50

for Rot_indx in tqdm(range(0,numRots)):

    RanRot = MCMCsmp.rndm_orth_matrix(NumParmsEx3)

    ex3MCMCSmpBaseRot = [RanRot @ A for A in ex3MCMCSmpBase]

    #Generate Histogram
    #Dimensions For Histogram Plot


    R = 5
    dr = .1

    histFileNm = FileNmBase + "/Rotations/" +"Baseline_Histogram_Rot_" + str(Rot_indx) + ".png"
    makeHistGrid(R, dr, ex3MCMCSmpBaseRot, NumParmsEx3,histFileNm, True)

In [ ]:
#Experiment 3: rho v p

#Input to studies
#[p,NumRho,NumSamples]

#ImpLst = [[10,20,10000],[100,20,10000],[1000,20,5000],[10000,10,1000]]
ImpLst = [[10,20,20000],[100,20,10000],[1000,20,5000],[10000,20,1000]]
#ImpLst = [[10,10,500],[100,10,500]]


for Imp in ImpLst:
    pCur = Imp[0]

    NumRho = Imp[1]
    delRho = 1/NumRho
    rho = delRho
    NumSamps = Imp[2]
    
    print("Currently running: p=" + str(pCur))
    print("Delta rho: " + str(delRho))
    print("Number of Samples: " + str(NumSamps))

    rhoLst = []
    ESSLstOG = []
    ESSLstLoc = []
    ESSLstGlob = []
    MSJDLstOG = []
    MSJDLstLoc = []
    MSJDLstGlob = []

    for parmIndx in range(0,NumParmsEx3):
        ESSLstOG.append([])
        ESSLstLoc.append([])
        ESSLstGlob.append([])

    rhoCur= rho
    for curRnInx in range(0,NumRho-1):
        rhoLst.append(rhoCur)
        rhoCur += delRho

    if __name__ == "__main__": 
        MCMCsampRun = []
        with ProcessPoolExecutor(max_workers=mp.cpu_count()) as pool:
            for rhoCur in rhoLst:
                strIndx = random.randint(math.floor(.95* MCMCpoolLen), MCMCpoolLen)
                q0z = ex3MCMCSmpBase[strIndx]
                
                CurfNinput = (q0z,NumParmsEx3,CovEx3,rhoCur,PotEx3Pass,pCur,NumSamps)
                MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsmPCN,*CurfNinput))
                MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsmlocPCNMTM,*CurfNinput))
                MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsBBMTM,*CurfNinput))

            print("Total MCMC Runs: " + str(len(MCMCsampRun)))
            results = [None]*len(MCMCsampRun)
            # map Future -> index so we can keep ordered results
            where = {f:i for i, f in enumerate(MCMCsampRun)}

            for f in tqdm(as_completed(MCMCsampRun), total=len(MCMCsampRun), desc="Parallel MCMC Runs"):
                i = where[f]
                results[i] = f.result()
    
    
    for curRnInx in tqdm(range(0,NumRho-1), desc= "Compiling Data"):
        curSampOGData = MCMCsampRun[3*curRnInx].result()
        curSampLocData = MCMCsampRun[3*curRnInx +1].result()
        curSampGlobData = MCMCsampRun[3*curRnInx +2].result()

        for parmIndx in range(0,NumParmsEx3):    
            ESSLstOG[parmIndx].append(curSampOGData[0][parmIndx])
            ESSLstLoc[parmIndx].append(curSampLocData[0][parmIndx])
            ESSLstGlob[parmIndx].append(curSampGlobData[0][parmIndx])
        
        MSJDLstOG.append(curSampOGData[1])
        MSJDLstLoc.append(curSampLocData[1])
        MSJDLstGlob.append(curSampGlobData[1])

    curRunData ="p_" + str(pCur) + "_drho_" + str(delRho) + "_NSamps_" + str(NumSamps)
    csvFileNm = FileNmBase + curRunData +  "_DATA.csv"

    writeCSV(csvFileNm,[rhoLst,ESSLstOG,ESSLstLoc,ESSLstGlob,MSJDLstOG,MSJDLstLoc,MSJDLstGlob])
    

    
    #Generate figures for ESS/N vs rho for each component of the posterior
    for parmIndx in range(0,NumParmsEx3):
        fig, ax = plt.subplots(figsize=(6, 4))

        ax.plot(rhoLst, ESSLstOG[parmIndx], linestyle="-", label=r"$mpCN$", color="tab:orange")
        ax.plot(rhoLst, ESSLstLoc[parmIndx], linestyle="-", label=r"$mpCNlocMTM$", color="tab:green")
        ax.plot(rhoLst, ESSLstGlob[parmIndx], linestyle="-",label=r"$mpCNMTM$", color="tab:blue")

        ax.set_xlabel(r"$\rho$")
        ax.set_ylabel(r"$ESS/N$")
        ax.set_title(r"Mixing as measured by ESS/N for p ="+str(pCur)+" for Parameter Index " + str(parmIndx))
        ax.grid(alpha=0.3)
        ax.legend()

        plt.tight_layout()
        plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_ParaIndx_" + str(parmIndx)+ ".png")
        plt.close(fig)
    
    
    fig_colors = generate_colors(NumParmsEx3)
    
    #Generate figures for ESS/N comparing different components for mpCN
    fig, ax = plt.subplots(figsize=(6, 4))
    for parmIndx in range(0,NumParmsEx3):
        ax.plot(rhoLst, ESSLstOG[parmIndx], linestyle="-", label="cmp_"+str(parmIndx), color=fig_colors[parmIndx])
    
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(r"$ESS/N$")
    ax.set_title(r"Mixing for mpCN as measured by ESS/N for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_mpCN.png")
    plt.close(fig)
    
    #Generate figures for ESS/N comparing different components for mpCNMTM
    fig, ax = plt.subplots(figsize=(6, 4))
    for parmIndx in range(0,NumParmsEx3):
        ax.plot(rhoLst, ESSLstGlob[parmIndx], linestyle="-", label="cmp_"+str(parmIndx), color=fig_colors[parmIndx])
    
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(r"$ESS/N$")
    ax.set_title(r"Mixing for mpCN_MTM as measured by ESS/N for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_mpCN_MTM.png")
    plt.close(fig)
    
    #Generate figures for ESS/N comparing different components for mpCNMTMloc
    fig, ax = plt.subplots(figsize=(6, 4))
    for parmIndx in range(0,NumParmsEx3):
        ax.plot(rhoLst, ESSLstLoc[parmIndx], linestyle="-", label="cmp_"+str(parmIndx), color=fig_colors[parmIndx])
    
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(r"$ESS/N$")
    ax.set_title(r"Mixing for mpCN_loc_MTM as measured by ESS/N for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_mpCN_MTM_loc.png")
    plt.close(fig)
    
    fig, ax = plt.subplots(figsize=(6, 4))

    ax.plot(rhoLst, MSJDLstOG, linestyle="-",label=r"$mpCN$", color="tab:orange")
    ax.plot(rhoLst, MSJDLstLoc, linestyle="-",label=r"$mpCNlocMTM$", color="tab:green")
    ax.plot(rhoLst, MSJDLstGlob, linestyle="-",label=r"$mpCNMTM$", color="tab:blue")

    ax.set_xlabel(r"rho")
    ax.set_ylabel(r"MSJD")
    ax.set_title(r"Mixing as measured by MSJD for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_MSJD_v_rho.png")
    plt.close(fig)


In [ ]:
#Experiment 4: Set-up

#This experiment is based on Model Problem B

#We Specifying Problem Parameters as follows
#Model dimension and parameter space size are
ModDmEx4 = 4
NumParmsEx4 = int(ModDmEx4*(ModDmEx4 -1)/2)

#This time we set g = [1,...,1] and z = kap^{-1} g which is
#leads to a stiff log likihood of the form 
#Pot(A) = kap^{-1}||(A - kap I)^{-1} A||^2
expId = 1 #Experiment ID
kapEx4 = 0.3 #Make kap smaller to get a stiffer problem


#expId = 2 #Experiment ID
#kapEx4 = 0.1 #Make kap smaller to get a stiffer problem

gvecEx4 = np.ones(ModDmEx4)
sigEx4 = 0.5

zEx4 = MCMCsmp.getThA(ModDmEx4, np.zeros(NumParmsEx4), gvecEx4, kapEx4)

dataDimEx4 = ModDmEx4


CovEx4 = np.array([[ 3.09615255, -0.42256446, -0.01699562, 1.52032843, 0.196684, 0.91198176],
 [-0.42256446,  0.47653539,  0.13517459, -0.15567356, -0.12246178,  0.03620587],
 [-0.01699562,  0.13517459,  0.90420916,  0.35872565,  0.0865228,   0.42293649],
 [ 1.52032843, -0.15567356,  0.35872565,  2.1249358,   0.03393446,  1.34534855],
 [ 0.196684,   -0.12246178,  0.0865228,   0.03393446,  0.90437014, -0.111162  ],
 [ 0.91198176,  0.03620587,  0.42293649,  1.34534855, -0.111162,    1.63623487]])

#cov0 = 5
#gam = 1.5
#CovDiag = [cov0* (j**(-gam)) for j in list(range(1,NumParmsEx4+1))]
#Q = MCMCsmp.rndm_orth_matrix(NumParmsEx4)
#CovEx4 = Q.T@ MCMCsmp.mkDiagCov(CovDiag)@ Q

#Make Problem Information File
FileNmBase = "Data/Large_p_study/Experiment_4/" + "Ex_ID_"+ str(expId) + "/"
os.makedirs(FileNmBase, exist_ok=True)

probDataFile = FileNmBase + "Problem_Info.txt"
with open(probDataFile, 'w') as file:
    file.write("Target Type:  Model Problem B \n")
    file.write("Model Dim: " + str(NumParmsEx4) + "\n")
    file.write("g: " + str(gvecEx4) + "\n")
    file.write("z: " + str(zEx4) + "\n")
    file.write("sig: " + str(sigEx4)+ "\n")
    file.write("kappa: " + str(kapEx4)+ "\n")
    file.write("Prior Cov: " + str(CovEx4) + "\n")

#Make Loglikehood function with lambda workaround
#using PotExAD(a, gvec, sig, ModDm, z, kap, dataDim, mode = None):
#print(inspect.signature(MCMCsmp.PotExAD))


PotEx4Pass = partial(MCMCsmp.PotExAD, gvec=gvecEx4, sig=sigEx4, ModDm=ModDmEx4, z=zEx4, kap=kapEx4, dataDim=dataDimEx4, mode = "soft")

L = 100000
stffMEx4, stffVEx4 = MCMCsmp.DetStiffness(NumParmsEx4,PotEx4Pass,CovEx4,L)

#for k in range(0,10):
#    stffMEx4, stffVEx4 = MCMCsmp.DetStiffness(NumParmsEx4,PotEx4Pass,CovEx4,L)
#    print("Mean: " + str(stffMEx4))
#    print("Var: " + str(stffVEx4) + "\n")

with open(probDataFile, 'a') as file:
    file.write("\n")
    file.write("Problem Stiffness: \n")
    file.write("M := int Pot(x) mu0(dx): " + str(stffMEx4) + "\n")
    file.write("V := int (Pot(x) -M)^2 mu0(dx): " + str(stffVEx4) + "\n")
    


In [ ]:
#Experiment 4
#Perform a baseline/warm-up MCMC run

q0 = np.zeros(NumParmsEx4)
rhoWarm = .3
pWarm = 100

numSmpWarm = 5000
burninWarm = 4000

WarmSamps = MCMCsmp.MpCN(q0,NumParmsEx4,CovEx4,rhoWarm,PotEx4Pass,pWarm,numSmpWarm)

NumRuns =  100 #of total runs
NumSamps = 10000 #samples per run

#NumRuns =  10 #of total runs
#NumSamps = 10000 #samples per run

if __name__ == "__main__": 
    
    with ProcessPoolExecutor(max_workers=mp.cpu_count()) as pool:
        MCMCsampRun = []
        for run in range(0,NumRuns):
            strIndx = random.randint(burninWarm, numSmpWarm)
            q0zCur = WarmSamps[strIndx]
            CurfNinput = (q0zCur,NumParmsEx4,CovEx4,rhoWarm,PotEx4Pass,pWarm,numSmpWarm)
            MCMCsampRun.append(pool.submit(MCMCsmp.MpCN,*CurfNinput))


        print("Total MCMC Runs: " + str(len(MCMCsampRun)))
        results = [None]*len(MCMCsampRun)
        # map Future -> index so we can keep ordered results
        where = {f:i for i, f in enumerate(MCMCsampRun)}

        for f in tqdm(as_completed(MCMCsampRun), total=len(MCMCsampRun), desc="Parallel MCMC Runs"):
            i = where[f]
            results[i] = f.result()
    csvFileNm = FileNmBase + "Baseline_Samples.csv"
    for curRnInx in tqdm(range(0,len(MCMCsampRun)), desc= "Compiling Data"):
         writeCSV(csvFileNm,MCMCsampRun[curRnInx].result())

#Generate Histogram
#Dimensions For Histogram Plot

ex4MCMCSmpBase = readCSV(csvFileNm)
MCMCpoolLen = len(ex4MCMCSmpBase)
print("Number of MpCN Samples Now Available: " + str(MCMCpoolLen))

R = 5
dr = .1

histFileNm = FileNmBase + "Baseline_Histogram.png"
makeHistGrid(R, dr, ex4MCMCSmpBase, NumParmsEx4,histFileNm, True)

In [ ]:
##Generate rotations of the posterior for visualization

os.makedirs(FileNmBase + "/Rotations", exist_ok=True)

numRots = 5

for Rot_indx in tqdm(range(0,numRots)):

    RanRot = MCMCsmp.rndm_orth_matrix(NumParmsEx4)

    ex4MCMCSmpBaseRot = [RanRot @ A for A in ex4MCMCSmpBase]

    #Generate Histogram
    #Dimensions For Histogram Plot


    R = 5
    dr = .1

    histFileNm = FileNmBase + "/Rotations/" +"Baseline_Histogram_Rot_" + str(Rot_indx) + ".png"
    makeHistGrid(R, dr, ex4MCMCSmpBaseRot, NumParmsEx4,histFileNm, True)

In [ ]:
#Experiment 4: rho v p

csvFileNm = FileNmBase + "Baseline_Samples.csv"
ex4MCMCSmpBase = readCSV(csvFileNm)
MCMCpoolLen = len(ex4MCMCSmpBase)

#Input to studies
#[p,NumRho,NumSamps]
#p: value to p to run study
#NumRho: 1/NumRho Specifies the step size for the study
#NumSamps: Length of the MCMC at each rho value

#ImpLst = [[1000,20,100000],[10000,20,20000],[100000, 20,5000],[10,20,10000000],[100,20,5000000]]
ImpLst = [[100,20,200000],[1000,20,50000],[10000,20,10000]]
#ImpLst = [[1000,20,100000],[10000,20,20000],[100000, 20,5000],[10,20,10000000],[100,20,5000000]]

for Imp in ImpLst:
    pCur = Imp[0]

    NumRho = Imp[1]
    delRho = 1/NumRho
    rho = delRho
    NumSamps = Imp[2]
    
    print("Currently running: p=" + str(pCur))
    print("Delta rho: " + str(delRho))
    print("Number of Samples: " + str(NumSamps))

    rhoLst = []
    ESSLstOG = []
    ESSLstLoc = []
    ESSLstGlob = []
    MSJDLstOG = []
    MSJDLstLoc = []
    MSJDLstGlob = []

    for parmIndx in range(0,NumParmsEx4):
        ESSLstOG.append([])
        ESSLstLoc.append([])
        ESSLstGlob.append([])

    rhoCur= rho
    for curRnInx in range(0,NumRho-1):
        rhoLst.append(rhoCur)
        rhoCur += delRho

    if __name__ == "__main__": 
        MCMCsampRun = []
        with ProcessPoolExecutor(max_workers=mp.cpu_count()) as pool:
            for rhoCur in rhoLst:
                strIndx = random.randint(math.floor(.95* MCMCpoolLen), MCMCpoolLen)
                q0z = ex4MCMCSmpBase[strIndx]
                
                CurfNinput = (q0z,NumParmsEx4,CovEx4,rhoCur,PotEx4Pass,pCur,NumSamps)
                MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsmPCN,*CurfNinput))
                MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsmlocPCNMTM,*CurfNinput))
                MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsBBMTM,*CurfNinput))

            print("Total MCMC Runs: " + str(len(MCMCsampRun)))
            results = [None]*len(MCMCsampRun)
            # map Future -> index so we can keep ordered results
            where = {f:i for i, f in enumerate(MCMCsampRun)}

            for f in tqdm(as_completed(MCMCsampRun), total=len(MCMCsampRun), desc="Parallel MCMC Runs"):
                i = where[f]
                results[i] = f.result()
    
    
    for curRnInx in tqdm(range(0,NumRho-1), desc= "Compiling Data"):
        curSampOGData = MCMCsampRun[3*curRnInx].result()
        curSampLocData = MCMCsampRun[3*curRnInx +1].result()
        curSampGlobData = MCMCsampRun[3*curRnInx +2].result()

        for parmIndx in range(0,NumParmsEx4):    
            ESSLstOG[parmIndx].append(curSampOGData[0][parmIndx])
            ESSLstLoc[parmIndx].append(curSampLocData[0][parmIndx])
            ESSLstGlob[parmIndx].append(curSampGlobData[0][parmIndx])
        
        MSJDLstOG.append(curSampOGData[1])
        MSJDLstLoc.append(curSampLocData[1])
        MSJDLstGlob.append(curSampGlobData[1])

    curRunData ="p_" + str(pCur) + "_drho_" + str(delRho) + "_NSamps_" + str(NumSamps)
    csvFileNm = FileNmBase + curRunData +  "_DATA.csv"

    writeCSV(csvFileNm,[rhoLst,ESSLstOG,ESSLstLoc,ESSLstGlob,MSJDLstOG,MSJDLstLoc,MSJDLstGlob])
    

    
    #Generate figures for ESS/N vs rho for each component of the posterior
    for parmIndx in range(0,NumParmsEx4):
        fig, ax = plt.subplots(figsize=(6, 4))

        ax.plot(rhoLst, ESSLstOG[parmIndx], linestyle="-", label=r"$mpCN$", color="tab:orange")
        ax.plot(rhoLst, ESSLstLoc[parmIndx], linestyle="-", label=r"$mpCNlocMTM$", color="tab:green")
        ax.plot(rhoLst, ESSLstGlob[parmIndx], linestyle="-",label=r"$mpCNMTM$", color="tab:blue")

        ax.set_xlabel(r"$\rho$")
        ax.set_ylabel(r"$ESS/N$")
        ax.set_title(r"Mixing as measured by ESS/N for p ="+str(pCur)+" for Parameter Index " + str(parmIndx))
        ax.grid(alpha=0.3)
        ax.legend()

        plt.tight_layout()
        plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_ParaIndx_" + str(parmIndx)+ ".png")
        plt.close(fig)
    
    
    fig_colors = generate_colors(NumParmsEx4)
    
    #Generate figures for ESS/N comparing different components for mpCN
    fig, ax = plt.subplots(figsize=(6, 4))
    for parmIndx in range(0,NumParmsEx4):
        ax.plot(rhoLst, ESSLstOG[parmIndx], linestyle="-", label="cmp_"+str(parmIndx), color=fig_colors[parmIndx])
    
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(r"$ESS/N$")
    ax.set_title(r"Mixing for mpCN as measured by ESS/N for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_mpCN.png")
    plt.close(fig)
    
    #Generate figures for ESS/N comparing different components for mpCNMTM
    fig, ax = plt.subplots(figsize=(6, 4))
    for parmIndx in range(0,NumParmsEx4):
        ax.plot(rhoLst, ESSLstGlob[parmIndx], linestyle="-", label="cmp_"+str(parmIndx), color=fig_colors[parmIndx])
    
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(r"$ESS/N$")
    ax.set_title(r"Mixing for mpCN_MTM as measured by ESS/N for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_mpCN_MTM.png")
    plt.close(fig)
    
    #Generate figures for ESS/N comparing different components for mpCNMTMloc
    fig, ax = plt.subplots(figsize=(6, 4))
    for parmIndx in range(0,NumParmsEx4):
        ax.plot(rhoLst, ESSLstLoc[parmIndx], linestyle="-", label="cmp_"+str(parmIndx), color=fig_colors[parmIndx])
    
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(r"$ESS/N$")
    ax.set_title(r"Mixing for mpCN_loc_MTM as measured by ESS/N for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_mpCN_MTM_loc.png")
    plt.close(fig)
    
    fig, ax = plt.subplots(figsize=(6, 4))

    ax.plot(rhoLst, MSJDLstOG, linestyle="-",label=r"$mpCN$", color="tab:orange")
    ax.plot(rhoLst, MSJDLstLoc, linestyle="-",label=r"$mpCNlocMTM$", color="tab:green")
    ax.plot(rhoLst, MSJDLstGlob, linestyle="-",label=r"$mpCNMTM$", color="tab:blue")

    ax.set_xlabel(r"rho")
    ax.set_ylabel(r"MSJD")
    ax.set_title(r"Mixing as measured by MSJD for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_MSJD_v_rho.png")
    plt.close(fig)

In [ ]:
#Experiment 4: Set-up

#This experiment is based on Model Problem B

#We Specifying Problem Parameters as follows
#Model dimension and parameter space size are
ModDmEx4 = 4
NumParmsEx4 = int(ModDmEx4*(ModDmEx4 -1)/2)

#This time we set g = [1,...,1] and z = kap^{-1} g which is
#leads to a stiff log likihood of the form 
#Pot(A) = kap^{-1}||(A - kap I)^{-1} A||^2
expId = 1 #Experiment ID
kapEx4 = 0.3 #Make kap smaller to get a stiffer problem


#expId = 2 #Experiment ID
#kapEx4 = 0.1 #Make kap smaller to get a stiffer problem

gvecEx4 = np.ones(ModDmEx4)
sigEx4 = 0.5

zEx4 = MCMCsmp.getThA(ModDmEx4, np.zeros(NumParmsEx4), gvecEx4, kapEx4)

dataDimEx4 = ModDmEx4


CovEx4 = np.array([[ 3.09615255, -0.42256446, -0.01699562, 1.52032843, 0.196684, 0.91198176],
 [-0.42256446,  0.47653539,  0.13517459, -0.15567356, -0.12246178,  0.03620587],
 [-0.01699562,  0.13517459,  0.90420916,  0.35872565,  0.0865228,   0.42293649],
 [ 1.52032843, -0.15567356,  0.35872565,  2.1249358,   0.03393446,  1.34534855],
 [ 0.196684,   -0.12246178,  0.0865228,   0.03393446,  0.90437014, -0.111162  ],
 [ 0.91198176,  0.03620587,  0.42293649,  1.34534855, -0.111162,    1.63623487]])

#cov0 = 5
#gam = 1.5
#CovDiag = [cov0* (j**(-gam)) for j in list(range(1,NumParmsEx4+1))]
#Q = MCMCsmp.rndm_orth_matrix(NumParmsEx4)
#CovEx4 = Q.T@ MCMCsmp.mkDiagCov(CovDiag)@ Q

#Make Problem Information File
FileNmBase = "Data/Large_p_study/Experiment_4/" + "Ex_ID_"+ str(expId) + "/"
os.makedirs(FileNmBase, exist_ok=True)

probDataFile = FileNmBase + "Problem_Info.txt"
with open(probDataFile, 'w') as file:
    file.write("Target Type:  Model Problem B \n")
    file.write("Model Dim: " + str(NumParmsEx4) + "\n")
    file.write("g: " + str(gvecEx4) + "\n")
    file.write("z: " + str(zEx4) + "\n")
    file.write("sig: " + str(sigEx4)+ "\n")
    file.write("kappa: " + str(kapEx4)+ "\n")
    file.write("Prior Cov: " + str(CovEx4) + "\n")

#Make Loglikehood function with lambda workaround
#using PotExAD(a, gvec, sig, ModDm, z, kap, dataDim, mode = None):
#print(inspect.signature(MCMCsmp.PotExAD))


PotEx4Pass = partial(MCMCsmp.PotExAD, gvec=gvecEx4, sig=sigEx4, ModDm=ModDmEx4, z=zEx4, kap=kapEx4, dataDim=dataDimEx4, mode = "soft")

L = 100000
stffMEx4, stffVEx4 = MCMCsmp.DetStiffness(NumParmsEx4,PotEx4Pass,CovEx4,L)

#for k in range(0,10):
#    stffMEx4, stffVEx4 = MCMCsmp.DetStiffness(NumParmsEx4,PotEx4Pass,CovEx4,L)
#    print("Mean: " + str(stffMEx4))
#    print("Var: " + str(stffVEx4) + "\n")

with open(probDataFile, 'a') as file:
    file.write("\n")
    file.write("Problem Stiffness: \n")
    file.write("M := int Pot(x) mu0(dx): " + str(stffMEx4) + "\n")
    file.write("V := int (Pot(x) -M)^2 mu0(dx): " + str(stffVEx4) + "\n")
    


In [ ]:
#Experiment 4
#Perform a baseline/warm-up MCMC run

q0 = np.zeros(NumParmsEx4)
rhoWarm = .3
pWarm = 100

numSmpWarm = 5000
burninWarm = 4000

WarmSamps = MCMCsmp.MpCN(q0,NumParmsEx4,CovEx4,rhoWarm,PotEx4Pass,pWarm,numSmpWarm)

NumRuns =  100 #of total runs
NumSamps = 10000 #samples per run

#NumRuns =  10 #of total runs
#NumSamps = 10000 #samples per run

if __name__ == "__main__": 
    
    with ProcessPoolExecutor(max_workers=mp.cpu_count()) as pool:
        MCMCsampRun = []
        for run in range(0,NumRuns):
            strIndx = random.randint(burninWarm, numSmpWarm)
            q0zCur = WarmSamps[strIndx]
            CurfNinput = (q0zCur,NumParmsEx4,CovEx4,rhoWarm,PotEx4Pass,pWarm,numSmpWarm)
            MCMCsampRun.append(pool.submit(MCMCsmp.MpCN,*CurfNinput))


        print("Total MCMC Runs: " + str(len(MCMCsampRun)))
        results = [None]*len(MCMCsampRun)
        # map Future -> index so we can keep ordered results
        where = {f:i for i, f in enumerate(MCMCsampRun)}

        for f in tqdm(as_completed(MCMCsampRun), total=len(MCMCsampRun), desc="Parallel MCMC Runs"):
            i = where[f]
            results[i] = f.result()
    csvFileNm = FileNmBase + "Baseline_Samples.csv"
    for curRnInx in tqdm(range(0,len(MCMCsampRun)), desc= "Compiling Data"):
         writeCSV(csvFileNm,MCMCsampRun[curRnInx].result())

#Generate Histogram
#Dimensions For Histogram Plot

ex4MCMCSmpBase = readCSV(csvFileNm)
MCMCpoolLen = len(ex4MCMCSmpBase)
print("Number of MpCN Samples Now Available: " + str(MCMCpoolLen))

R = 5
dr = .1

histFileNm = FileNmBase + "Baseline_Histogram.png"
makeHistGrid(R, dr, ex4MCMCSmpBase, NumParmsEx4,histFileNm, True)

In [ ]:
#Experiment 5


#This experiment is based on Model Problem B

#Currently based on a data from a lost experiment to find z and g (which have been retained)

#We Specifying Problem Parameters as follows
#Model dimension and parameter space size are
ModDmEx5 = 4
NumParmsEx5 = int(ModDmEx5*(ModDmEx5 -1)/2)

kapEx5 = 0.01 #Make kap smaller to get a stiffer problem
sigEx5 = 0.01
gam = 1.1
CovDiag = [cov0* (j**(-gam)) for j in list(range(1,NumParmsEx5+1))]
#Q = MCMCsmp.rndm_orth_matrix(NumParmsEx4)
#CovEx4 = Q.T@ MCMCsmp.mkDiagCov(CovDiag)@ Q
CovEx5 = MCMCsmp.mkDiagCov(CovDiag)


#The idea is repeatedly sample from {A_j}_{j \geq 1} the prior \mu_0 = N(0,C)
#{g_j}_{j \geq 1} from N(0, I) and random rotations {Q_j}_{j \geq 1}
#For each (A_j,g_j we check if |\theta(A_j,g) - \theta(Q_j^* A_J Q_j, g)| < \epsilon
#and ||C^{-1/2} A ||- ||C^{-1/2} Q_j^* A_J Q_j||  < \epsilon and quit when we find such an A


def FindMatchFull(numParm, modDm, datacomp, Cov, kap, tol, lam):
    curErr = 2
    IddGo = MCMCsmp.mkDiagCov([1,1,0,0,0,0,0])
    mnA = np.zeros(numParm)
    mnGo = np.zeros(modDm)
    CovInv = np.linalg.inv(Cov)
    numtries = 1
    while curErr > tol:
        Acur = np.random.multivariate_normal(mnA, Cov,1)[0]
        Gcur = np.random.multivariate_normal(mnGo, IddGo,1)[0]
        #Qcur = MCMCsmp.rndm_orth_matrix(numParm)
        tAcur = -1*lam* Acur
        curAerr = np.linalg.norm((MCMCsmp.getThA(modDm, tAcur, Gcur, kap) - MCMCsmp.getThA(modDm, Acur, Gcur, kap))*datacomp) 
        rotInBulk = max(0,np.dot(CovInv @ tAcur, tAcur) -np.dot(CovInv @ Acur, Acur))
        curErr = min(curAerr + rotInBulk,curErr)
        numtries = numtries+ 1
        if numtries % 100000 == 0:
            print("Indx: " + str(numtries))
            print("A error: " +str(curAerr))
            print("Expansion: "+ str(rotInBulk))
            print("Running Error" + str(curErr))
    return Acur, Gcur, tAcur, numtries

#datacomp2nd4th = np.array([1,1,0,0,0,0,0])
#Atru, gvecEx5, tA, n= FindMatchFull(NumParmsEx5, ModDmEx5, datacomp2nd4th, CovEx5, kapEx5, .00001, .9)

gvecEx5 = np.array([-0.31350811, -0.04480832,  0.070213,   -0.04616241])
zEx5 = np.array([ 0.02366996,-0.,0.,-0.05559005])


FileNmBase = "Data/Large_p_study/Experiment_5/" + "Ex_ID_"+ str(expId) + "/"
os.makedirs(FileNmBase, exist_ok=True)

#probDataFile = FileNmBase + "Problem_Info.txt"
#with open(probDataFile, 'w') as file:
#    file.write("Target Type:  Model Problem B \n")
#    file.write("Model Dim: " + str(NumParmsEx5) + "\n")
#    file.write("g: " + str(gvecEx5) + "\n")
#    file.write("z: " + str(zEx5*datacomp2nd4th) + "\n")
#    file.write("sig: " + str(sigEx5)+ "\n")
#    file.write("kappa: " + str(kapEx5)+ "\n")
#    file.write("Prior Cov: " + str(CovEx5) + "\n")


#Careful to save olde data???
expId =  17 #Experiment ID
FileNmBase = "Data/Large_p_study/Experiment_5/" + "Ex_ID_"+ str(expId) + "/"
os.makedirs(FileNmBase, exist_ok=True)
csvFileNm = FileNmBase + "Baseline_Samples.csv"




#Generate Histogram
#Dimensions For Histogram Plot

ex5MCMCSmpBase = readCSV(csvFileNm)
MCMCpoolLen = len(ex6MCMCSmpBase)
print("Number of MpCN Samples Now Available: " + str(MCMCpoolLen))

R = 7
dr = .1


histFileNm = FileNmBase + "Baseline_Histogram.png"
makeHistGrid(R, dr, ex5MCMCSmpBase, NumParmsEx5,histFileNm, CovEx5, 0.95, True)


In [ ]:
#Experiment 6

#Gaussian target (changing Covariance/mean on `low modes')

#Model dim
TarDim6 = 5

cov0 = 4
gam = 2
CovDiag_p = [cov0* (j**(-gam)) for j in list(range(1,TarDim6+1))]
#Q = MCMCsmp.rndm_orth_matrix(NumParmsEx4)
#CovEx4 = Q.T@ MCMCsmp.mkDiagCov(CovDiag)@ Q
CovPriorEx6 = MCMCsmp.mkDiagCov(CovDiag_p)
print(CovPriorEx6)

#Perturb dim
pertDim6 = 2
#CovDiag_post = [.1*CovDiag_p[0], 10*CovDiag_p[1]]
CovDiag_post = [CovDiag_p[1], CovDiag_p[0]] + CovDiag_p[pertDim6:]
CovPostEx6 = MCMCsmp.mkDiagCov(CovDiag_post)
print(CovPostEx6)

meanPostEx6 = np.zeros(pertDim6)


#Make Problem Information File
expId =  5
FileNmBase = "Data/Large_p_study/Experiment_6/" + "Ex_ID_"+ str(expId) + "/"
os.makedirs(FileNmBase, exist_ok=True)

probDataFile = FileNmBase + "Problem_Info.txt"
with open(probDataFile, 'w') as file:
    file.write("Target Type:  Model Problem C \n")
    file.write("Target Dimension: " + str(TarDim6) + "\n")
    file.write("Prior Covariance: " + str(CovPriorEx6) + "\n")
    file.write("Perturbation Dimension: " + str(pertDim6) + "\n")
    file.write("Posterior Mean: " + str(meanPostEx6) + "\n")
    file.write("Posterior Covariance: " + str(CovPostEx6)+ "\n")

PriorCovInvEx6 = MCMCsmp.mkDiagCov([CovDiag_p[0]**(-1),CovDiag_p[1]**(-1)])
PostCovInvEx6 = MCMCsmp.mkDiagCov([CovDiag_post[0]**(-1),CovDiag_post[1]**(-1)])

PotEx6Pass = partial(MCMCsmp.PotGaussPert, TarDim=TarDim6, PertDim=pertDim6, PriorCovInv=PriorCovInvEx6, PostMean=meanPostEx6, PostCovInv=PostCovInvEx6, mode = "soft")

#Testing Potential
#print(PostCovInvEx6 - PriorCovInvEx6)   
#priorSmpLst = np.random.multivariate_normal(np.zeros(TarDim6), CovPriorEx6, size=5)
#print([MCMCsmp.PotGaussPert(priorSmp,TarDim6,pertDim6, PriorCovInvEx6,meanPostEx6,PostCovInvEx6) for priorSmp in priorSmpLst])
#print([-1/2*x[0:pertDim6].T @(PostCovInvEx6- PriorCovInvEx6)@x[0:pertDim6]  for x in priorSmpLst])
#CovPriorEx6Ext = MCMCsmp.mkDiagCov(CovDiag_post + CovDiag_p[pertDim6:])
#print(CovPriorEx6)
#print(CovPriorEx6Ext)
#print([-1/2*x.T @(np.linalg.inv(CovPriorEx6Ext)- np.linalg.inv(CovPriorEx6))@x  for x in priorSmpLst])


In [ ]:
#Experiment 6
#Perform a baseline/warm-up MCMC run

q0 = np.zeros(TarDim6)
rhoWarm = .95
pWarm = 100

numSmpWarm = 100000
burninWarm = 80000

#WarmSamps = MCMCsmp.MpCN(q0,TarDim6,CovPriorEx6,rhoWarm,PotEx6Pass,pWarm,numSmpWarm)
WarmSamps = MCMCsmp.locMpCNMTM(q0,TarDim6,CovPriorEx6,rhoWarm,PotEx6Pass,pWarm,numSmpWarm)



NumRuns =  100 #of total runs
NumSamps = 10000 #samples per run

#NumRuns =  10 #of total runs
#NumSamps = 10000 #samples per run

if __name__ == "__main__": 
    
    with ProcessPoolExecutor(max_workers=mp.cpu_count()) as pool:
        MCMCsampRun = []
        for run in range(0,NumRuns):
            strIndx = random.randint(burninWarm, numSmpWarm)
            q0zCur = WarmSamps[strIndx]
            CurfNinput = (q0zCur,TarDim6,CovPriorEx6,rhoWarm,PotEx6Pass,pWarm,numSmpWarm)
            #MCMCsampRun.append(pool.submit(MCMCsmp.MpCN,*CurfNinput))
            MCMCsampRun.append(pool.submit(MCMCsmp.locMpCNMTM,*CurfNinput))


        print("Total MCMC Runs: " + str(len(MCMCsampRun)))
        results = [None]*len(MCMCsampRun)
        # map Future -> index so we can keep ordered results
        where = {f:i for i, f in enumerate(MCMCsampRun)}

        for f in tqdm(as_completed(MCMCsampRun), total=len(MCMCsampRun), desc="Parallel MCMC Runs"):
            i = where[f]
            results[i] = f.result()
    csvFileNm = FileNmBase + "Baseline_Samples.csv"
    for curRnInx in tqdm(range(0,len(MCMCsampRun)), desc= "Compiling Data"):
         writeCSV(csvFileNm,MCMCsampRun[curRnInx].result())

#Generate Histogram
#Dimensions For Histogram Plot

ex6MCMCSmpBase = readCSV(csvFileNm)
MCMCpoolLen = len(ex6MCMCSmpBase)
print("Number of MpCN Samples Now Available: " + str(MCMCpoolLen))

R = 5
dr = .1

histFileNm = FileNmBase + "Baseline_Histogram.png"
makeHistGrid(R, dr, ex6MCMCSmpBase, TarDim6,histFileNm, CovPriorEx6, 0.95, True)



In [ ]:
rhotest =.5
ptest = 50
invChainLn = 500000
lag = 200
numChains = 50

if __name__ == "__main__": 
    with ProcessPoolExecutor(max_workers=mp.cpu_count()) as pool:
        MCMCsampRun = []
        for run in range(0,numChains):
            q0z = np.random.multivariate_normal(np.zeros(TarDim6), CovPostEx6, size=1)

            CurfNinput = (q0z,TarDim6,CovPriorEx6,rhotest,PotEx6Pass,ptest,invChainLn)
            MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsmPCNSingle,*CurfNinput))

        print("Total MCMC Runs: " + str(len(MCMCsampRun)))
        results = [None]*len(MCMCsampRun)
        # map Future -> index so we can keep ordered results
        where = {f:i for i, f in enumerate(MCMCsampRun)}

        for f in tqdm(as_completed(MCMCsampRun), total=len(MCMCsampRun), desc="Parallel MCMC Runs"):
            i = where[f]
            results[i] = f.result()


ESS_Data = []
for curRnInx in tqdm(range(0,numChains-1), desc= "Compiling Data"):
    ESS_Data.append(MCMCsampRun[curRnInx].result()[0][0])

# Make a histogram of the ESS_Data
plt.figure()
plt.hist(ESS_Data, bins='auto', edgecolor='black')
plt.xlabel('Value')
plt.ylabel('Count')
plt.title('Histogram of Computed ESS')
plt.grid(alpha=0.3)
plt.show()

# Compute the running average A = [y_1, ..., y_n]
A = []
running_sum = 0.0
for k, x in enumerate(ESS_Data, start=1):  # k goes 1..n
    running_sum += x
    y_k = running_sum / k          # y_k = (1/k) * sum_{j=1}^k x_j
    A.append(y_k)

# Plot y_k vs k
k_values = list(range(1, len(ESS_Data) + 1))

plt.figure()
plt.plot(k_values, A, marker='o')
plt.xlabel('k')
plt.ylabel('y_k (running average)')
plt.title('Running average vs index k')
plt.grid(True)
plt.show()

In [ ]:
rhotest =.5
ptest = 50
invChainLn = 500000
numChains = 50

if __name__ == "__main__": 
    with ProcessPoolExecutor(max_workers=mp.cpu_count()) as pool:
        MCMCsampRun = []
        for run in range(0,numChains):
            q0z = np.random.multivariate_normal(np.zeros(TarDim6), CovPostEx6, size=1)

            CurfNinput = (q0z,TarDim6,CovPriorEx6,rhotest,PotEx6Pass,ptest,invChainLn)
            MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsmPCNSingle,*CurfNinput))

        print("Total MCMC Runs: " + str(len(MCMCsampRun)))
        results = [None]*len(MCMCsampRun)
        # map Future -> index so we can keep ordered results
        where = {f:i for i, f in enumerate(MCMCsampRun)}

        for f in tqdm(as_completed(MCMCsampRun), total=len(MCMCsampRun), desc="Parallel MCMC Runs"):
            i = where[f]
            results[i] = f.result()


ESS_Data = []
for curRnInx in tqdm(range(0,numChains-1), desc= "Compiling Data"):
    ESS_Data.append(MCMCsampRun[curRnInx].result()[0][0])

# Make a histogram of the ESS_Data
plt.figure()
plt.hist(ESS_Data, bins='auto', edgecolor='black')
plt.xlabel('Value')
plt.ylabel('Count')
plt.title('Histogram of Computed ESS')
plt.grid(alpha=0.3)
plt.show()

# Compute the running average A = [y_1, ..., y_n]
A = []
running_sum = 0.0
for k, x in enumerate(ESS_Data, start=1):  # k goes 1..n
    running_sum += x
    y_k = running_sum / k          # y_k = (1/k) * sum_{j=1}^k x_j
    A.append(y_k)

# Plot y_k vs k
k_values = list(range(1, len(ESS_Data) + 1))

plt.figure()
plt.plot(k_values, A, marker='o')
plt.xlabel('k')
plt.ylabel('y_k (running average)')
plt.title('Running average vs index k')
plt.grid(True)
plt.show()


In [ ]:
rhotest =.5
ptest = 50
invChainLn = 1000000
numChains = 25

if __name__ == "__main__": 
    with ProcessPoolExecutor(max_workers=mp.cpu_count()) as pool:
        MCMCsampRun = []
        for run in range(0,numChains):
            q0z = np.random.multivariate_normal(np.zeros(TarDim6), CovPostEx6, size=1)

            CurfNinput = (q0z,TarDim6,CovPriorEx6,rhotest,PotEx6Pass,ptest,invChainLn)
            MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsmPCN,*CurfNinput))

        print("Total MCMC Runs: " + str(len(MCMCsampRun)))
        results = [None]*len(MCMCsampRun)
        # map Future -> index so we can keep ordered results
        where = {f:i for i, f in enumerate(MCMCsampRun)}

        for f in tqdm(as_completed(MCMCsampRun), total=len(MCMCsampRun), desc="Parallel MCMC Runs"):
            i = where[f]
            results[i] = f.result()


ESS_Data = []
for curRnInx in tqdm(range(0,numChains-1), desc= "Compiling Data"):
    ESS_Data.append(MCMCsampRun[curRnInx].result()[0][0])

# Make a histogram of the ESS_Data
plt.figure()
plt.hist(ESS_Data, bins='auto', edgecolor='black')
plt.xlabel('Value')
plt.ylabel('Count')
plt.title('Histogram of Computed ESS')
plt.grid(alpha=0.3)
plt.show()

# Compute the running average A = [y_1, ..., y_n]
A = []
running_sum = 0.0
for k, x in enumerate(ESS_Data, start=1):  # k goes 1..n
    running_sum += x
    y_k = running_sum / k          # y_k = (1/k) * sum_{j=1}^k x_j
    A.append(y_k)

# Plot y_k vs k
k_values = list(range(1, len(ESS_Data) + 1))

plt.figure()
plt.plot(k_values, A, marker='o')
plt.xlabel('k')
plt.ylabel('y_k (running average)')
plt.title('Running average vs index k')
plt.grid(True)
plt.show()

In [ ]:
#Experiment 6: rho v p

csvFileNm = FileNmBase + "Baseline_Samples.csv"
ex6MCMCSmpBase = readCSV(csvFileNm)
MCMCpoolLen = len(ex6MCMCSmpBase)

#Input to studies
#[p,NumRho,NumSamps,10]
#p: value to p to run study
#NumRho: 1/NumRho Specifies the step size for the study
#NumSamps: Length of the MCMC at each rho value
#NumerChain: chains per rho value


ImpLst = [[10,20,20000,10]]


for Imp in ImpLst:
    pCur = Imp[0]

    NumRho = Imp[1]
    delRho = 1/NumRho
    rho = delRho
    NumSamps = Imp[2]
    
    print("Currently running: p=" + str(pCur))
    print("Delta rho: " + str(delRho))
    print("Number of Samples: " + str(NumSamps))

    rhoLst = []
    ESSLstOG = []
    ESSLstLoc = []
    ESSLstGlob = []
    MSJDLstOG = []
    MSJDLstLoc = []
    MSJDLstGlob = []

    for parmIndx in range(0,TarDim6):
        ESSLstOG.append([])
        ESSLstLoc.append([])
        ESSLstGlob.append([])

    rhoCur= rho
    for curRnInx in range(0,NumRho-1):
        rhoLst.append(rhoCur)
        rhoCur += delRho

    if __name__ == "__main__": 
        MCMCsampRun = []
        with ProcessPoolExecutor(max_workers=mp.cpu_count()) as pool:
            for rhoCur in rhoLst:
                q0z = np.random.multivariate_normal(np.zeros(TarDim6), CovPostEx6, size=Imp[3])

                CurfNinput = (q0z,TarDim6,CovPriorEx6,rhoCur,PotEx6Pass,pCur,NumSamps,NumChain)
                MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsmPCN,*CurfNinput))
                MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsmlocPCNMTM,*CurfNinput))
                MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsBBMTM,*CurfNinput))

            print("Total MCMC Runs: " + str(len(MCMCsampRun)))
            results = [None]*len(MCMCsampRun)
            # map Future -> index so we can keep ordered results
            where = {f:i for i, f in enumerate(MCMCsampRun)}

            for f in tqdm(as_completed(MCMCsampRun), total=len(MCMCsampRun), desc="Parallel MCMC Runs"):
                i = where[f]
                results[i] = f.result()
    
    
    for curRnInx in tqdm(range(0,NumRho-1), desc= "Compiling Data"):
        curSampOGData = MCMCsampRun[3*curRnInx].result()
        curSampLocData = MCMCsampRun[3*curRnInx +1].result()
        curSampGlobData = MCMCsampRun[3*curRnInx +2].result()

        for parmIndx in range(0,TarDim6):    
            ESSLstOG[parmIndx].append(curSampOGData[0][parmIndx])
            ESSLstLoc[parmIndx].append(curSampLocData[0][parmIndx])
            ESSLstGlob[parmIndx].append(curSampGlobData[0][parmIndx])
        
        MSJDLstOG.append(curSampOGData[1])
        MSJDLstLoc.append(curSampLocData[1])
        MSJDLstGlob.append(curSampGlobData[1])

    curRunData ="p_" + str(pCur) + "_drho_" + str(delRho) + "_NSamps_" + str(NumSamps)
    csvFileNm = FileNmBase + curRunData +  "_DATA.csv"

    writeCSV(csvFileNm,[rhoLst,ESSLstOG,ESSLstLoc,ESSLstGlob,MSJDLstOG,MSJDLstLoc,MSJDLstGlob])
    

    
    #Generate figures for ESS/N vs rho for each component of the posterior
    for parmIndx in range(0,TarDim6):
        fig, ax = plt.subplots(figsize=(6, 4))

        ax.plot(rhoLst, ESSLstOG[parmIndx], linestyle="-", label=r"$mpCN$", color="tab:orange")
        ax.plot(rhoLst, ESSLstLoc[parmIndx], linestyle="-", label=r"$mpCNlocMTM$", color="tab:green")
        ax.plot(rhoLst, ESSLstGlob[parmIndx], linestyle="-",label=r"$mpCNMTM$", color="tab:blue")

        ax.set_xlabel(r"$\rho$")
        ax.set_ylabel(r"$ESS/N$")
        ax.set_title(r"Mixing as measured by ESS/N for p ="+str(pCur)+" for Parameter Index " + str(parmIndx))
        ax.grid(alpha=0.3)
        ax.legend()

        plt.tight_layout()
        plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_ParaIndx_" + str(parmIndx)+ ".png")
        plt.close(fig)
    
    
    fig_colors = generate_colors(TarDim6)
    
    #Generate figures for ESS/N comparing different components for mpCN
    fig, ax = plt.subplots(figsize=(6, 4))
    for parmIndx in range(0,TarDim6):
        ax.plot(rhoLst, ESSLstOG[parmIndx], linestyle="-", label="cmp_"+str(parmIndx), color=fig_colors[parmIndx])
    
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(r"$ESS/N$")
    ax.set_title(r"Mixing for mpCN as measured by ESS/N for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_mpCN.png")
    plt.close(fig)
    
    #Generate figures for ESS/N comparing different components for mpCNMTM
    fig, ax = plt.subplots(figsize=(6, 4))
    for parmIndx in range(0,TarDim6):
        ax.plot(rhoLst, ESSLstGlob[parmIndx], linestyle="-", label="cmp_"+str(parmIndx), color=fig_colors[parmIndx])
    
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(r"$ESS/N$")
    ax.set_title(r"Mixing for mpCN_MTM as measured by ESS/N for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_mpCN_MTM.png")
    plt.close(fig)
    
    #Generate figures for ESS/N comparing different components for mpCNMTMloc
    fig, ax = plt.subplots(figsize=(6, 4))
    for parmIndx in range(0,TarDim6):
        ax.plot(rhoLst, ESSLstLoc[parmIndx], linestyle="-", label="cmp_"+str(parmIndx), color=fig_colors[parmIndx])
    
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(r"$ESS/N$")
    ax.set_title(r"Mixing for mpCN_loc_MTM as measured by ESS/N for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_mpCN_MTM_loc.png")
    plt.close(fig)
    
    fig, ax = plt.subplots(figsize=(6, 4))

    ax.plot(rhoLst, MSJDLstOG, linestyle="-",label=r"$mpCN$", color="tab:orange")
    ax.plot(rhoLst, MSJDLstLoc, linestyle="-",label=r"$mpCNlocMTM$", color="tab:green")
    ax.plot(rhoLst, MSJDLstGlob, linestyle="-",label=r"$mpCNMTM$", color="tab:blue")

    ax.set_xlabel(r"rho")
    ax.set_ylabel(r"MSJD")
    ax.set_title(r"Mixing as measured by MSJD for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_MSJD_v_rho.png")
    plt.close(fig)

In [ ]:
#Experiment 6: rho v p

csvFileNm = FileNmBase + "Baseline_Samples.csv"
ex6MCMCSmpBase = readCSV(csvFileNm)
MCMCpoolLen = len(ex6MCMCSmpBase)

#Input to studies
#[p,NumRho,NumSamps]
#p: value to p to run study
#NumRho: 1/NumRho Specifies the step size for the study
#NumSamps: Length of the MCMC at each rho value

ImpLst = [[10,100,5000000],[100,50,1000000],[1000,20,100000]]#,[10000,20,10000]]
#ImpLst = [[10,50,50000],[100,50,10000],[1000,20,10000],[10000,20,10000]]
#ImpLst = [[1000,20,100000],[10000,20,20000],[100000, 20,5000],[10,20,10000000],[100,20,5000000]]
#ImpLst = [[100,20,200000],[1000,20,50000],[10000,20,10000]]
#ImpLst = [[1000,20,100000],[10000,20,20000],[100000, 20,5000],[10,20,10000000],[100,20,5000000]]

for Imp in ImpLst:
    pCur = Imp[0]

    NumRho = Imp[1]
    delRho = 1/NumRho
    rho = delRho
    NumSamps = Imp[2]
    
    print("Currently running: p=" + str(pCur))
    print("Delta rho: " + str(delRho))
    print("Number of Samples: " + str(NumSamps))

    rhoLst = []
    ESSLstOG = []
    ESSLstLoc = []
    ESSLstGlob = []
    MSJDLstOG = []
    MSJDLstLoc = []
    MSJDLstGlob = []

    for parmIndx in range(0,TarDim6):
        ESSLstOG.append([])
        ESSLstLoc.append([])
        ESSLstGlob.append([])

    rhoCur= rho
    for curRnInx in range(0,NumRho-1):
        rhoLst.append(rhoCur)
        rhoCur += delRho

    if __name__ == "__main__": 
        MCMCsampRun = []
        with ProcessPoolExecutor(max_workers=mp.cpu_count()) as pool:
            for rhoCur in rhoLst:
                strIndx = random.randint(math.floor(.95* MCMCpoolLen), MCMCpoolLen)
                q0z = ex6MCMCSmpBase[strIndx]

                CurfNinput = (q0z,TarDim6,CovPriorEx6,rhoCur,PotEx6Pass,pCur,NumSamps)
                MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsmPCN,*CurfNinput))
                MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsmlocPCNMTM,*CurfNinput))
                MCMCsampRun.append(pool.submit(MCMCsmp.mixMetricsBBMTM,*CurfNinput))

            print("Total MCMC Runs: " + str(len(MCMCsampRun)))
            results = [None]*len(MCMCsampRun)
            # map Future -> index so we can keep ordered results
            where = {f:i for i, f in enumerate(MCMCsampRun)}

            for f in tqdm(as_completed(MCMCsampRun), total=len(MCMCsampRun), desc="Parallel MCMC Runs"):
                i = where[f]
                results[i] = f.result()
    
    
    for curRnInx in tqdm(range(0,NumRho-1), desc= "Compiling Data"):
        curSampOGData = MCMCsampRun[3*curRnInx].result()
        curSampLocData = MCMCsampRun[3*curRnInx +1].result()
        curSampGlobData = MCMCsampRun[3*curRnInx +2].result()

        for parmIndx in range(0,TarDim6):    
            ESSLstOG[parmIndx].append(curSampOGData[0][parmIndx])
            ESSLstLoc[parmIndx].append(curSampLocData[0][parmIndx])
            ESSLstGlob[parmIndx].append(curSampGlobData[0][parmIndx])
        
        MSJDLstOG.append(curSampOGData[1])
        MSJDLstLoc.append(curSampLocData[1])
        MSJDLstGlob.append(curSampGlobData[1])

    curRunData ="p_" + str(pCur) + "_drho_" + str(delRho) + "_NSamps_" + str(NumSamps)
    csvFileNm = FileNmBase + curRunData +  "_DATA.csv"

    writeCSV(csvFileNm,[rhoLst,ESSLstOG,ESSLstLoc,ESSLstGlob,MSJDLstOG,MSJDLstLoc,MSJDLstGlob])
    

    
    #Generate figures for ESS/N vs rho for each component of the posterior
    for parmIndx in range(0,TarDim6):
        fig, ax = plt.subplots(figsize=(6, 4))

        ax.plot(rhoLst, ESSLstOG[parmIndx], linestyle="-", label=r"$mpCN$", color="tab:orange")
        ax.plot(rhoLst, ESSLstLoc[parmIndx], linestyle="-", label=r"$mpCNlocMTM$", color="tab:green")
        ax.plot(rhoLst, ESSLstGlob[parmIndx], linestyle="-",label=r"$mpCNMTM$", color="tab:blue")

        ax.set_xlabel(r"$\rho$")
        ax.set_ylabel(r"$ESS/N$")
        ax.set_title(r"Mixing as measured by ESS/N for p ="+str(pCur)+" for Parameter Index " + str(parmIndx))
        ax.grid(alpha=0.3)
        ax.legend()

        plt.tight_layout()
        plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_ParaIndx_" + str(parmIndx)+ ".png")
        plt.close(fig)
    
    
    fig_colors = generate_colors(TarDim6)
    
    #Generate figures for ESS/N comparing different components for mpCN
    fig, ax = plt.subplots(figsize=(6, 4))
    for parmIndx in range(0,TarDim6):
        ax.plot(rhoLst, ESSLstOG[parmIndx], linestyle="-", label="cmp_"+str(parmIndx), color=fig_colors[parmIndx])
    
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(r"$ESS/N$")
    ax.set_title(r"Mixing for mpCN as measured by ESS/N for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_mpCN.png")
    plt.close(fig)
    
    #Generate figures for ESS/N comparing different components for mpCNMTM
    fig, ax = plt.subplots(figsize=(6, 4))
    for parmIndx in range(0,TarDim6):
        ax.plot(rhoLst, ESSLstGlob[parmIndx], linestyle="-", label="cmp_"+str(parmIndx), color=fig_colors[parmIndx])
    
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(r"$ESS/N$")
    ax.set_title(r"Mixing for mpCN_MTM as measured by ESS/N for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_mpCN_MTM.png")
    plt.close(fig)
    
    #Generate figures for ESS/N comparing different components for mpCNMTMloc
    fig, ax = plt.subplots(figsize=(6, 4))
    for parmIndx in range(0,TarDim6):
        ax.plot(rhoLst, ESSLstLoc[parmIndx], linestyle="-", label="cmp_"+str(parmIndx), color=fig_colors[parmIndx])
    
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(r"$ESS/N$")
    ax.set_title(r"Mixing for mpCN_loc_MTM as measured by ESS/N for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_ESS_v_rho_mpCN_MTM_loc.png")
    plt.close(fig)
    
    fig, ax = plt.subplots(figsize=(6, 4))

    ax.plot(rhoLst, MSJDLstOG, linestyle="-",label=r"$mpCN$", color="tab:orange")
    ax.plot(rhoLst, MSJDLstLoc, linestyle="-",label=r"$mpCNlocMTM$", color="tab:green")
    ax.plot(rhoLst, MSJDLstGlob, linestyle="-",label=r"$mpCNMTM$", color="tab:blue")

    ax.set_xlabel(r"rho")
    ax.set_ylabel(r"MSJD")
    ax.set_title(r"Mixing as measured by MSJD for p ="+str(pCur))
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FileNmBase + curRunData+ "_MSJD_v_rho.png")
    plt.close(fig)

In [ ]:
#TESTING

from arviz import ess

def msjd(samples: np.ndarray) -> float:
    """
    Computes MSJD Mean‑Squared Jumping Distances for a list of samples
    {x_0,x_1,..., x_m} as m^{-1} sum_{k=1}^{m} |x_{k-1} - x_k|^2
    The imputs are
        samples : array_like, shape (N, d)
    """
    diffs = np.diff(samples, axis=0)      # shape (N-1, d)
    return np.mean(np.sum(diffs**2, axis=1))

def mixMetricsmPCNMulti(q0str,NumParms,Cov,rho,Pot,p,totSamps,nmRun,saveLn):
    essMet = [[] for _ in range(nmRun)]
    MSJDMet =[]
    for run in range(nmRun):
        curSamp = MCMCsmp.MpCN(q0str[run],NumParms,Cov,rho,Pot,p,totSamps)
        for parmIndx in range(0,NumParms):    
            essMet[run].append(ess(curSamp[:,parmIndx])/totSamps)
        MSJDMet.append(msjd(curSamp))
    esslst = [sum(comp) / len(comp) for comp in essMet]
    MSJD = sum(MSJDMet)/len(MSJDMet)
    return esslst, MSJD, curSamp[0:saveLn]

NumSamps = 10000
NumChains = 3
rhoCur = .3
pCur = 10
SampleLnSv = 500

q0zSt = np.random.multivariate_normal(np.zeros(TarDim6), CovPostEx6, size=NumChains)

mixMetricsmPCNMulti(q0zSt,TarDim6,CovPriorEx6,rhoCur,PotEx6Pass,pCur,NumSamps,NumChains,SampleLnSv)